# Nanoplasma paper figures from reduced H5 — rebuilt path-safe version

This notebook restores the old paper-figure workflow, but removes the fragile hard-coded local imports and centralizes the paths at the top.

Workflow:

1. Use `01_export_reduced_h5.ipynb` to create the reduced H5 file.
2. Use this notebook to generate the paper figures from the reduced H5.
3. The `binningOpenPMD` leaving-particle diagnostic is not used here.


In [ ]:
# =========================================================
# 0) Imports, local package bootstrap, and path configuration
# =========================================================

%matplotlib inline

from pathlib import Path
import sys
import os
import re

import h5py
import numpy as np
import matplotlib.pyplot as plt
import scipy.constants as sc

# ---------------------------------------------------------
# Local Git package import without installing anything
# ---------------------------------------------------------
def find_repo_root(start=None, marker=("src", "nanoplasma_analysis")):
    """Find repo root by walking upward until src/nanoplasma_analysis exists."""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for p in [start, *start.parents]:
        candidate = p / marker[0] / marker[1]
        if candidate.exists() and candidate.is_dir():
            return p
    # Fallback for your current JUWELS-style location
    fallback = Path("/e/scratch/jureap18/medina2/PIConGPU_analisis")
    if (fallback / marker[0] / marker[1]).exists():
        return fallback
    raise FileNotFoundError(
        f"Could not find repo root from {start}. Expected src/nanoplasma_analysis."
    )

REPO = find_repo_root()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from nanoplasma_analysis import NanoPlasmaRun, extract_step_from_filename

print("Using repo:", REPO)
print("Using source:", SRC)

# ---------------------------------------------------------
# USER PATHS: edit these only if needed
# ---------------------------------------------------------
# Current simulation output folder.
SIM_OUTPUT = Path("/e/scratch/jureap18/medina2/004_V1_LowDensity/simOutput")

# Older fallback from the original paper notebooks.
SIM_OUTPUT_OLD = Path("/p/scratch/jureap18/medina2/2026_nanoplasma/004_Neutral_Smallsteps/simOutput")

if not SIM_OUTPUT.exists() and SIM_OUTPUT_OLD.exists():
    SIM_OUTPUT = SIM_OUTPUT_OLD

# Common places where reduced H5 files may exist.
H5_SEARCH_DIRS = [
    Path.cwd(),
    REPO,
    REPO / "data",
    SIM_OUTPUT,
    SIM_OUTPUT.parent,
    Path("/e/scratch/jureap18/medina2"),
    Path("/p/scratch/jureap18/medina2/2026_nanoplasma"),
    Path("/p/scratch/pwfa-trojan/medina2/2026_nanoplasma"),
]
H5_SEARCH_DIRS = [p for p in H5_SEARCH_DIRS if p.exists()]

# Common places where experimental reduced H5 files may exist.
EXP_SEARCH_DIRS = [
    Path.cwd(),
    REPO,
    REPO / "data",
    SIM_OUTPUT.parent,
    Path("/e/scratch/jureap18/medina2"),
    Path("/p/scratch/jureap18/medina2/2026_nanoplasma"),
    Path("/p/scratch/pwfa-trojan/medina2/Exp_data"),
]
EXP_SEARCH_DIRS = [p for p in EXP_SEARCH_DIRS if p.exists()]

FIG_DIR = Path.cwd() / "paper_figures"
FIG_DIR.mkdir(exist_ok=True)

print("SIM_OUTPUT:", SIM_OUTPUT)
print("H5 search dirs:")
for p in H5_SEARCH_DIRS:
    print("  ", p)
print("Figure output dir:", FIG_DIR)

# ---------------------------------------------------------
# H5 file discovery helpers
# ---------------------------------------------------------
def _first_existing(candidates):
    for c in candidates:
        c = Path(c)
        if c.exists():
            return c
    return None


def find_first_h5(patterns, search_dirs, required=False, label="H5"):
    """Find first H5 matching patterns in search dirs. Returns Path or None."""
    hits = []
    for d in search_dirs:
        for pat in patterns:
            hits.extend(sorted(d.glob(pat)))
    # prefer non-empty files and shorter names first only after direct order
    hits = [h for h in hits if h.exists()]
    if hits:
        return hits[0]
    if required:
        raise FileNotFoundError(
            f"Could not find {label}. Patterns={patterns}, dirs={search_dirs}"
        )
    return None


def as_h5_string(p):
    return None if p is None else str(Path(p))

# Main reduced H5 for simulation paper plots.
# Prefer your previous paper names; otherwise pick any *reduced*.h5 nearby.
H5_NEUTRAL_PATH = _first_existing([
    SIM_OUTPUT.parent / "Run004_reducedV2.h5",
    SIM_OUTPUT.parent / "Run004_reduced.h5",
    SIM_OUTPUT.parent / "Run001_reduced.h5",
]) or find_first_h5(
    ["Run004_reducedV2.h5", "Run004_reduced.h5", "Run001_reduced.h5", "*reduced*.h5"],
    H5_SEARCH_DIRS,
    required=False,
    label="main reduced H5",
)

# Optional log-reduced file used by one old spectrum cell.
H5_NEUTRAL_LOG_PATH = _first_existing([
    SIM_OUTPUT.parent / "Run004_reduced_log.h5",
]) or find_first_h5(
    ["Run004_reduced_log.h5", "*reduced_log*.h5"],
    H5_SEARCH_DIRS,
    required=False,
    label="log reduced H5",
)

# Optional pre-ionized file. Leave as None unless you have one.
H5_IONIZED_PATH = find_first_h5(
    ["*ionized*reduced*.h5", "*pre*reduced*.h5", "Run002_reduced*.h5"],
    H5_SEARCH_DIRS,
    required=False,
    label="pre-ionized reduced H5",
)

# Experimental files used by later optional comparison cells.
EXP_ION_H5_PATH = find_first_h5(
    ["all_runs_ion_kinematics.h5"],
    EXP_SEARCH_DIRS,
    required=False,
    label="experimental ion KEDI H5",
)

EXP_VMI_H5_PATH = find_first_h5(
    ["all_runs_vmi_radii.h5"],
    EXP_SEARCH_DIRS,
    required=False,
    label="experimental VMI radii H5",
)

H5_NEUTRAL = as_h5_string(H5_NEUTRAL_PATH)
H5_NEUTRAL_LOG = as_h5_string(H5_NEUTRAL_LOG_PATH)
H5_IONIZED = as_h5_string(H5_IONIZED_PATH)
EXP_ION_H5 = as_h5_string(EXP_ION_H5_PATH)
EXP_H5 = EXP_ION_H5
exp_vmi_h5 = as_h5_string(EXP_VMI_H5_PATH)

print("\nResolved files:")
print("  H5_NEUTRAL     =", H5_NEUTRAL)
print("  H5_NEUTRAL_LOG =", H5_NEUTRAL_LOG)
print("  H5_IONIZED     =", H5_IONIZED)
print("  EXP_ION_H5     =", EXP_ION_H5)
print("  EXP_VMI_H5     =", exp_vmi_h5)

if H5_NEUTRAL is None:
    print("\nWARNING: No reduced H5 found automatically.")
    print("Set H5_NEUTRAL manually, e.g. H5_NEUTRAL = '/path/to/Run004_reducedV2.h5'")

# ---------------------------------------------------------
# Generic H5 helpers used by all figures
# ---------------------------------------------------------
def require_file(path, label="file"):
    if path is None:
        raise FileNotFoundError(f"{label} is None. Set the path in the top config cell.")
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"{label} not found: {p}")
    return str(p)


def h5_inventory(path):
    path = require_file(path, "H5 file")
    with h5py.File(path, "r") as f:
        inv = {}
        for k in f.keys():
            inv[k] = list(f[k].keys()) if isinstance(f[k], h5py.Group) else []
        return inv


def h5_read(path, dset):
    path = require_file(path, "H5 file")
    with h5py.File(path, "r") as f:
        return np.array(f[dset])


def h5_has(path, dset):
    if path is None or not Path(path).exists():
        return False
    with h5py.File(path, "r") as f:
        return dset in f


def load_axes(path):
    path = require_file(path, "H5 file")
    axes = {}
    with h5py.File(path, "r") as f:
        for k in f["axes"].keys():
            axes[k] = np.array(f["axes"][k])
    return axes


def load_timeseries(path):
    path = require_file(path, "H5 file")
    ts = {}
    with h5py.File(path, "r") as f:
        for k in f["timeseries"].keys():
            ts[k] = np.array(f["timeseries"][k])
    return ts


def load_spectra(path):
    path = require_file(path, "H5 file")
    sp = {}
    with h5py.File(path, "r") as f:
        for k in f["spectra"].keys():
            sp[k] = np.array(f["spectra"][k])
    return sp


def load_radial(path):
    path = require_file(path, "H5 file")
    rd = {}
    with h5py.File(path, "r") as f:
        for k in f["radial"].keys():
            rd[k] = np.array(f["radial"][k])
    return rd


def list_slice_steps(path):
    path = require_file(path, "H5 file")
    with h5py.File(path, "r") as f:
        if "slices" not in f:
            return []
        return sorted(list(f["slices"].keys()))


def read_slice(path, step_key, name):
    dset = f"slices/{step_key}/{name}"
    return h5_read(path, dset)


def parse_step(step_key):
    # "step_00190000" -> 190000
    return int(step_key.split("_", 1)[1])


def available_slice_fields(path, step_key):
    path = require_file(path, "H5 file")
    with h5py.File(path, "r") as f:
        return list(f["slices"][step_key].keys())


## 0) Sanity check: what’s in the files

In [ ]:
for label, p in [("Neutral", H5_NEUTRAL), ("Neutral log", H5_NEUTRAL_LOG), ("Pre-ionized", H5_IONIZED)]:
    if p is None:
        print(f"\n=== {label} ===")
        print("not set / not found")
        continue
    print("\n===", label, "===")
    inv = h5_inventory(p)
    for k in inv:
        print(f"{k:12s} -> {inv[k][:12]}{' ...' if len(inv[k])>12 else ''}")


# Figure 1 — Storyboard maps from `/slices`

This reproduces your “movie condensed” panel.  
It uses whatever slice datasets are available per step. Because exporter conventions can differ, this cell tries common names and prints what it finds.

**Tip:** Use exactly the same selected steps for both runs for clean side-by-side figures.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import scipy.constants as sc

# =======================
# USER KNOBS (EDIT THESE)
# =======================

N_FRAMES = 6
I0_Wcm2 = 1e14
FIELD_VLIM = 2.0
LOGNORM_VMIN = 1e-5  # on normalized density (0..1)

# crop padding (extra pixels added after centered crop)
PAD_XY = 0
PAD_XZ = 0

# ---- MANUAL CENTERS (pixel coordinates) ----
# For Ex_xy_SI (array is [y, x])
CX_XY = 510
CY_XY = 580

# For rho_*_xz_SI (array is [z, x])
CX_XZ = 510
CZ_XZ = 550

# ---- MANUAL ZOOM (half sizes in pixels) ----
# XY crop will be (2*HALF_XY+1) x (2*HALF_XY+1)
HALF_XY = 100

# XZ crop will be (2*HALF_XZ_X+1) x (2*HALF_XZ_Z+1)  (x width, z height)
HALF_XZ_X = 400
HALF_XZ_Z = 400

# =======================
# helpers
# =======================

def get_pixel_size_nm_from_meta(path):
    """Return (dx_nm, dy_nm, dz_nm) if stored in /meta, else None."""
    import h5py
    with h5py.File(path, "r") as f:
        if "meta" not in f:
            return None
        m = f["meta"]

        def read_any(cands):
            for c in cands:
                if c in m:
                    return float(np.array(m[c]))
            return None

        Dx = read_any(["Dx_SI", "cell_width_SI", "dx_SI"])
        Dy = read_any(["Dy_SI", "cell_height_SI", "dy_SI"])
        Dz = read_any(["Dz_SI", "cell_depth_SI", "dz_SI"])

    if Dx is None or Dy is None or Dz is None:
        return None
    return Dx * 1e9, Dy * 1e9, Dz * 1e9  # nm


def set_ticks_nm(ax, shape, dx_nm=1, dy_nm=1, nticks=4, which="x", xlabel="x", ylabel="y"):
    """Ticks in nm if dx_nm/dy_nm provided; else pixel index ticks."""
    ny, nx = shape

    if which in ("x", "both"):
        xs = np.linspace(0, nx - 1, nticks).round().astype(int)
        ax.set_xticks(xs)
        if dx_nm is not None:
            ax.set_xticklabels([f"{x * dx_nm:.0f}" for x in xs])
            ax.set_xlabel(f"{xlabel} (nm)")
        else:
            ax.set_xticklabels([str(x) for x in xs])
            ax.set_xlabel(f"{xlabel} (px)")

    if which in ("y", "both"):
        ys = np.linspace(0, ny - 1, nticks).round().astype(int)
        ax.set_yticks(ys)
        if dy_nm is not None:
            ax.set_yticklabels([f"{y * dy_nm:.0f}" for y in ys])
            ax.set_ylabel(f"{ylabel} (nm)")
        else:
            ax.set_yticklabels([str(y) for y in ys])
            ax.set_ylabel(f"{ylabel} (px)")


def pick_even(step_keys, n_frames=6):
    step_keys = list(step_keys)
    if len(step_keys) <= n_frames:
        return step_keys
    idx = np.linspace(0, len(step_keys) - 1, n_frames).round().astype(int)
    idx = np.unique(idx)
    while len(idx) < n_frames:
        missing = [i for i in range(len(step_keys)) if i not in idx]
        if not missing:
            break
        idx = np.sort(np.concatenate([idx, [missing[len(missing) // 2]]]))
    return [step_keys[i] for i in idx[:n_frames]]


def bbox_centered(shape, cx, cy, halfx, halfy, pad=0):
    """Return (x0,x1,y0,y1) for array shape (ny,nx)."""
    ny, nx = shape
    x0 = max(0, int(round(cx - halfx - pad)))
    x1 = min(nx - 1, int(round(cx + halfx + pad)))
    y0 = max(0, int(round(cy - halfy - pad)))
    y1 = min(ny - 1, int(round(cy + halfy + pad)))
    return (x0, x1, y0, y1)


def crop2(A, bbox):
    x0, x1, y0, y1 = bbox
    return A[y0 : y1 + 1, x0 : x1 + 1]


def get_times_for_slices(path, sel_steps):
    """Try to map sel_steps -> time_fs from axes/time_fs if available."""
    all_steps = list_slice_steps(path)
    try:
        axes = load_axes(path)
        t = axes.get("time_fs", None)
    except Exception:
        t = None

    if t is not None and len(t) == len(all_steps):
        m = {sk: float(t[i]) for i, sk in enumerate(all_steps)}
        return [m.get(sk, np.nan) for sk in sel_steps]
    return [np.nan] * len(sel_steps)


# =======================
# main plot
# =======================

def plot_storyboard_horizontal_paper_manual_center(path, title, n_frames=6):
    all_steps = list_slice_steps(path)
    sel_steps = pick_even(all_steps, n_frames=n_frames)
    t_list = get_times_for_slices(path, sel_steps)

    # check required datasets exist
    f0 = available_slice_fields(path, sel_steps[0])
    need = ("Ex_xy_SI", "rho_e_xz_SI", "rho_i_xz_SI")
    for k in need:
        if k not in f0:
            raise RuntimeError(f"Missing {k} in slices/{sel_steps[0]}. Available: {f0}")

    # load frames (planes are fixed!)
    Ex_list, rhoe_list, rhoi_list = [], [], []
    for sk in sel_steps:
        Ex_list.append(np.squeeze(read_slice(path, sk, "Ex_xy_SI")))       # XY (y,x)
        rhoe_list.append(np.squeeze(read_slice(path, sk, "rho_e_xz_SI")))  # XZ (z,x)
        rhoi_list.append(np.squeeze(read_slice(path, sk, "rho_i_xz_SI")))  # XZ (z,x)

    # normalize field Ex/E0
    E0 = np.sqrt(2.0 * I0_Wcm2 * 1e4 / (sc.c * sc.epsilon_0))
    Exn_list = [A / (E0 + 1e-300) for A in Ex_list]

    # normalize densities to global max across frames (then LogNorm in plot)
    max_rhoe = np.nanmax([np.nanmax(np.abs(A)) for A in rhoe_list]) + 1e-300
    max_rhoi = np.nanmax([np.nanmax(np.abs(A)) for A in rhoi_list]) + 1e-300
    rhoeN = []
    rhoiN = []
    for A in rhoe_list:
        B = np.abs(A) / max_rhoe
        B[B <= 0] = np.nan
        rhoeN.append(B)
    for A in rhoi_list:
        B = np.abs(A) / max_rhoi
        B[B <= 0] = np.nan
        rhoiN.append(B)

    rho_norm = LogNorm(vmin=LOGNORM_VMIN, vmax=1.0)

    # ---- centered crops (FIXED, STABLE) ----
    bbox_xy = bbox_centered(Exn_list[0].shape, CX_XY, CY_XY, HALF_XY, HALF_XY, pad=PAD_XY)
    bbox_xz = bbox_centered(rhoiN[0].shape,  CX_XZ, CZ_XZ, HALF_XZ_X, HALF_XZ_Z, pad=PAD_XZ)

    Exn_list = [crop2(A, bbox_xy) for A in Exn_list]   # XY crop
    rhoeN    = [crop2(A, bbox_xz) for A in rhoeN]      # XZ crop
    rhoiN    = [crop2(A, bbox_xz) for A in rhoiN]      # XZ crop

    # style
    plt.rcParams.update({
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.titlesize": 11,
        "axes.labelsize": 11,
    })

    # geometry
    ncols = len(sel_steps)
    fig = plt.figure(figsize=(2.0 * ncols + 1.25, 6.2))
    gs = fig.add_gridspec(
        nrows=3, ncols=ncols + 1,
        width_ratios=[1] * ncols + [0.1],
        wspace=0.0, hspace=0.00
    )

    # pixel size for tick labels (optional)
    pix = get_pixel_size_nm_from_meta(path)
    dx_nm = pix[0] if pix else None
    dy_nm = pix[1] if pix else None
    dz_nm = pix[2] if pix else None

    ims_E, ims_e, ims_i = [], [], []
    last_axE = last_axNe = last_axNi = None

    for j in range(ncols):
        ttl = f"{t_list[j]:.0f} fs" if np.isfinite(t_list[j]) else f"{parse_step(sel_steps[j])}"

        axE  = fig.add_subplot(gs[0, j])
        axNe = fig.add_subplot(gs[1, j])
        axNi = fig.add_subplot(gs[2, j])
        last_axE, last_axNe, last_axNi = axE, axNe, axNi

        imE = axE.imshow(
            Exn_list[j],
            cmap="RdBu_r", origin="lower",
            interpolation="nearest",
            vmin=-FIELD_VLIM, vmax=FIELD_VLIM,
            aspect="equal",
        )
        imNe = axNe.imshow(
            rhoeN[j],
            cmap="coolwarm", origin="lower",
            interpolation="nearest",
            norm=rho_norm,
            aspect="equal",
        )
        imNi = axNi.imshow(
            rhoiN[j],
            cmap="coolwarm", origin="lower",
            interpolation="nearest",
            norm=rho_norm,
            aspect="equal",
        )

        ims_E.append(imE); ims_e.append(imNe); ims_i.append(imNi)

        # frame styling
        for ax in (axE, axNe, axNi):
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_visible(True)
                sp.set_linewidth(0.5)
                sp.set_color("black")

        axE.set_title(ttl)

        # labels on left column only
        if j == 0:
            axE.set_ylabel(r"$E_x/E_0$")
            axNe.set_ylabel(r"$|\rho_e|$")
            axNi.set_ylabel(r"$|\rho_i|$")

            # y ticks only in first column
            set_ticks_nm(axE,  Exn_list[j].shape, dx_nm=1, dy_nm=1, which="y", ylabel="z")
            set_ticks_nm(axNe, rhoeN[j].shape,    dx_nm=1, dy_nm=1, which="y", ylabel="y")
            set_ticks_nm(axNi, rhoiN[j].shape,    dx_nm=1, dy_nm=1, which="y", ylabel="y")

        # bottom row x ticks for all columns (matches your current behavior)
        set_ticks_nm(axNi, rhoiN[j].shape, dx_nm=1, dy_nm=1, which="x", xlabel="x")

        # hide bottom tick labels for inner columns if you want less clutter
        #if j != 0 or j !=-1:
         #   axNi.tick_params(labelbottom=False)

    # colorbars (one per row)
    caxE = fig.add_subplot(gs[0, -1])
    caxe = fig.add_subplot(gs[1, -1])
    caxi = fig.add_subplot(gs[2, -1])

    cbE = fig.colorbar(ims_E[-1], cax=caxE)
    cbe = fig.colorbar(ims_e[-1], cax=caxe)
    cbi = fig.colorbar(ims_i[-1], cax=caxi)

    cbE.set_label(r"$E_x/E_0$")
    cbe.set_label("norm.")
    cbi.set_label("norm.")

    # match colorbar heights to axes heights
    bboxE  = last_axE.get_position()
    bboxNe = last_axNe.get_position()
    bboxNi = last_axNi.get_position()
    for cax, bbox in [(caxE, bboxE), (caxe, bboxNe), (caxi, bboxNi)]:
        p = cax.get_position()
        cax.set_position([p.x0, bbox.y0, p.width, bbox.height])

    fig.suptitle(title, y=0.995, fontsize=13)
    plt.savefig("Fig_1.png", dpi=600, bbox_inches="tight")  
    plt.show()


# =======================
# RUN
# =======================

# plot_storyboard_horizontal_paper_manual_center(H5_IONIZED, "Pre-ionized run — storyboard", n_frames=N_FRAMES)
plot_storyboard_horizontal_paper_manual_center(H5_NEUTRAL, "Neutral run — storyboard", n_frames=N_FRAMES)


# Figure 2 — Ignition + collective dynamics (timeseries + radial)

Panels you can do from reduced H5:

- $N_e(t)$ and $dN_e/dt$ (electron yield / avalanche signature)
- Ion charge state: $\langle Z \rangle(t)$ and charge fractions
- Expansion: $\langle r \rangle(t)$ and/or $R_{99}(t)$ from ion radial distribution
- Net charge inside droplet vs time (approximate, see notes)



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------- user knobs ----------
TAU_FWHM_FS = 40.0          # laser intensity FWHM (fs)
NORM_RATE_SMOOTH = 7        # odd integer window for simple moving-average smoothing of dNe/dt
SHOW_Z = (0, 1, 2)          # Z lines to show in middle panel
# -------------------------------

def movavg(y, w):
    w = int(w)
    if w < 3 or w % 2 == 0:
        return y
    k = w // 2
    ypad = np.pad(y, (k, k), mode="edge")
    return np.convolve(ypad, np.ones(w)/w, mode="valid")

def gaussian_envelope(t_fs, fwhm_fs):
    # normalized intensity envelope, peak at t=0
    return np.exp(-4*np.log(2) * (t_fs / fwhm_fs)**2)

def ddt_midpoints(t_fs, y):
    t = np.asarray(t_fs)
    y = np.asarray(y)
    dt = np.diff(t)
    dy = np.diff(y)
    tmid = 0.5*(t[1:] + t[:-1])
    return tmid, dy/(dt + 1e-300)

def plot_figure2_ignition(path, title):
    axes = load_axes(path)
    ts = load_timeseries(path)
    rd = load_radial(path)

    t = axes["time_fs"]

    # ---------- TOP: electron population (normalized) + creation rate (background) ----------
    if "He_e_N_real" in ts:
        Ne = ts["He_e_N_real"].astype(float)
    else:
        Ne = ts["He_e_N_macro"].astype(float)

    Ne_norm = Ne / (np.nanmax(Ne) + 1e-300)
    tmid, dNe_dt = ddt_midpoints(t, Ne_norm)
    dNe_dt = movavg(dNe_dt, NORM_RATE_SMOOTH)
    # normalize rate for a nice filled background
    dNe_dt_norm = dNe_dt / (np.nanmax(np.abs(dNe_dt)) + 1e-300)

    # ---------- MIDDLE: ion charge populations Z=0,1,2 (normalized) + laser envelope background ----------
    if "He_i_charge_frac" not in ts:
        raise RuntimeError("Missing timeseries/He_i_charge_frac in reduced H5. "
                           "You need this for Z=0/1/2 ionization evolution.")
    fracZ = ts["He_i_charge_frac"]  # expected shape (Nt, Zmax+1)
    if fracZ.ndim != 2:
        raise RuntimeError(f"Unexpected He_i_charge_frac shape: {fracZ.shape}")

    # ensure rows correspond to time
    if fracZ.shape[0] != len(t) and fracZ.shape[1] == len(t):
        fracZ = fracZ.T

    # ---------- BOTTOM: ion expansion R99/R0 ----------
    r_mid = axes["r_mid_m"]
    n_r = rd["He_i_n_r"]
    # orient (Nt, Nr)
    if n_r.shape[0] == len(t) and n_r.shape[1] == len(r_mid):
        n_t_r = n_r
    elif n_r.shape[1] == len(t) and n_r.shape[0] == len(r_mid):
        n_t_r = n_r.T
    else:
        raise RuntimeError(f"He_i_n_r shape {n_r.shape} does not match (Nt,Nr)=({len(t)},{len(r_mid)})")

    n_t_r = np.maximum(n_t_r, 0.0)
    cdf = np.cumsum(n_t_r, axis=1)
    tot = cdf[:, -1] + 1e-300
    frac = cdf / tot[:, None]
    R99 = np.array([r_mid[min(np.searchsorted(frac[i], 0.99), len(r_mid)-1)] for i in range(len(t))])
    R0 = R99[0] if R99[0] > 0 else (np.nanmax(R99) + 1e-300)
    R99_over_R0 = R99 / (R0 + 1e-300)

    # ---------- figure styling ----------
    plt.rcParams.update({
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
    })

    fig, ax = plt.subplots(3, 1, figsize=(5,8), sharex=True,
                           gridspec_kw={"hspace": 0.0})

    # ---- TOP panel ----
     # Electron population (left axis)
    Ne_line = ax[0].plot(t, Ne_norm, ".-",lw=2.4, label=r"$N_{e^-}$ (norm.)")
    color_Ne = Ne_line[0].get_color()

    ax[0].set_ylabel(f"Ne$^-$ (norm.)")
    ax[0].set_ylim(0.7, 1.05)
    ax[0].grid(True, alpha=0.25)

    # Electron creation rate (right axis)
    ax_rate = ax[0].twinx()
    # normalize rate independently for visual clarity
    dNe_dt_norm2 = dNe_dt / (np.nanmax(np.abs(dNe_dt)) + 1e-300)
    ax_rate.fill_between(tmid,dNe_dt_norm2,0,color=color_Ne,alpha=0.22,lw=0,label="dNe/dt (norm.)"    )
    ax_rate.plot(tmid,dNe_dt_norm2,color=color_Ne,alpha=0.6,lw=1.2)
    ax_rate.set_ylabel("dNe/dt (norm.)", color=color_Ne)
    ax_rate.tick_params(axis="y", colors=color_Ne)
    ax_rate.spines["right"].set_color(color_Ne)
    ax_rate.set_ylim(-0.05, 1.05)
    # cleaner look
    ax_rate.spines["top"].set_visible(False)
    ax[0].legend(loc="center right",frameon=False)   
    
   # ---- MIDDLE panel ----
    # Ion charge populations on left axis
    for Z in SHOW_Z:
        if Z < fracZ.shape[1]:
            ax[1].plot(t, fracZ[:, Z], ".-", lw=2.2, label=f"Z={Z}")
    
    ax[1].set_ylabel("Ion charge pop. (frac.)")
    ax[1].set_ylim(-0.05, 1.05)
    ax[1].grid(True, alpha=0.25)
    ax[1].legend(loc="center right", frameon=False)
    # Laser envelope on right axis
    ax_laser = ax[1].twinx()
    laser_color = "gray"
    
    # Smooth time grid
    t_dense = np.linspace(np.nanmin(t), np.nanmax(t), 1500)
    
    # Gaussian intensity envelope
    I_dense = gaussian_envelope(t_dense, TAU_FWHM_FS)
    
    # convert to ponderomotive energy Up(t)
    lambda_um = 0.8
    Up_dense = 9.33e-14 * (I0_Wcm2 * I_dense) * (lambda_um**2)
    
    ax_laser.fill_between(t_dense, Up_dense, 0, color="gray", alpha=0.18, lw=0)
    ax_laser.plot(t_dense, Up_dense, color="gray", alpha=0.6, lw=1.2)
    
    ax_laser.set_ylabel(r"$U_p$ (eV)", color=laser_color)
    ax_laser.set_ylim(bottom=0)

    # ---- BOTTOM panel: R_rms/R0 (ions) + <n_e>/ncrit (inside R_rms) ----
    LAMBDA_NM = 800.0  # set to your laser wavelength
    
    def orient_to_t_r(A, Nt, Nr):
        A = np.asarray(A)
        if A.shape == (Nt, Nr):
            return A
        if A.shape == (Nr, Nt):
            return A.T
        raise RuntimeError(f"Expected (Nt,Nr)=({Nt},{Nr}) or (Nr,Nt), got {A.shape}")
    
    Nr = len(r_mid)
    Nt = len(t)
    
    n_i_t_r = orient_to_t_r(rd["He_i_n_r"], Nt, Nr)
    n_e_t_r = orient_to_t_r(rd["He_e_n_r"], Nt, Nr)
    
    n_i_t_r = np.maximum(n_i_t_r, 0.0)
    n_e_t_r = np.maximum(n_e_t_r, 0.0)
    
    # --- Ion rms radius ---
    Ni = np.sum(n_i_t_r, axis=1) + 1e-300
    r2_mean = np.sum(n_i_t_r * (r_mid[None, :]**2), axis=1) / Ni
    Rrms = np.sqrt(np.maximum(r2_mean, 0.0))
    
    R0 = Rrms[0] if Rrms[0] > 0 else (np.nanmax(Rrms) + 1e-300)
    Rrms_over_R0 = Rrms / (R0 + 1e-300)
    
    # --- Electron mean density inside Rrms ---
    # electrons inside Rrms: sum shells with r_mid <= Rrms(t)
    Ne_in = np.zeros(Nt)
    for i in range(Nt):
        mask = r_mid <= Rrms[i]
        Ne_in[i] = np.sum(n_e_t_r[i, mask])
    
    V_sphere = (4.0/3.0) * np.pi * (Rrms**3 + 1e-300)
    ne_mean = Ne_in / V_sphere  # 1/m^3
    
    # --- Critical density ---
    lam_m = LAMBDA_NM * 1e-9
    omega = 2.0*np.pi*sc.c/lam_m
    ncrit = sc.epsilon_0 * sc.m_e * omega**2 / sc.e**2  # 1/m^3
    
    ne_over_ncrit = ne_mean / (ncrit + 1e-300)
    
    # Plot left axis: radius
    ax[2].plot(t, Rrms_over_R0, lw=2.2, label=r"$R_{\rm rms}/R_0$")
    ax[2].set_ylabel(r"$R_{\rm rms}/R_0$")
    ax[2].grid(True, alpha=0.25)
    
    # Plot right axis: density in units of ncrit
    ax_den = ax[2].twinx()
    den_color = "red"
    ax_den.plot(t, ne_over_ncrit, lw=1.8, color=den_color, label=r"$\langle n_e\rangle/n_{\rm crit}$")
    #ax_den.set_yscale("log")
    ax_den.set_ylabel(r"$\langle n_e\rangle/n_{\rm crit}$", color=den_color)
    ax_den.tick_params(axis="y", colors=den_color)
    ax_den.spines["right"].set_color(den_color)
    ax_den.spines["top"].set_visible(False)
    ax[2].set_xlabel("time (fs) (t=0 at laser peak)")
    
    # Combined legend (bottom)
    ax[2].legend(
        [ax[2].lines[-1], ax_den.lines[-1]],
        [r"$R_{\rm rms}/R_0$", r"$\langle n_e\rangle/n_{\rm crit}$"],
        loc="center left",
        frameon=False
    )
    fig.suptitle(title, y=0.995)
    plt.tight_layout()
    plt.show()

# Run both cases
#plot_figure2_ignition(H5_IONIZED, "Pre-ionized: ignition + dynamics")
plot_figure2_ignition(H5_NEUTRAL, "Neutral: ignition + dynamics")


In [ ]:
# Removed stale old-cell expression: plot_ignition_dynamics


# Figure 3 — Angular anisotropy / asymmetry vs time (from dN/dμ)

From your reduced H5 you can compute a clean **forward-backward asymmetry** along the chosen axis from the stored `He_e_dNdmu(t,μ)`.

Define:
- forward = μ>0
- backward = μ<0
- $A(t) = (N_+ - N_-)/(N_+ + N_-)$

This is a strong “collective” diagnostic without needing raw momenta.

**Note:** the *phase-shift* analysis (dipole vs laser E) is not possible unless you export a dipole moment time series.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

def asymmetry_from_dNdmu(mu_mid, dNdmu_t_mu):
    H = np.asarray(dNdmu_t_mu)
    if H.shape[1] == len(mu_mid):
        Ht = H
    elif H.shape[0] == len(mu_mid):
        Ht = H.T
    else:
        raise ValueError("dNdmu shape doesn't match mu_mid.")
    mu = np.asarray(mu_mid)
    plus = mu > 0
    minus = mu < 0
    Np = np.sum(Ht[:, plus], axis=1)
    Nm = np.sum(Ht[:, minus], axis=1)
    return (Np - Nm) / (Np + Nm + 1e-300)

def _orient_mu_t(dNdmu, mu):
    H = np.asarray(dNdmu)
    # return (Nmu, Nt) for imshow with extent [tmin,tmax, mumin,mumax]
    if H.shape[0] == len(mu):
        return H
    if H.shape[1] == len(mu):
        return H.T
    raise ValueError(f"Cannot orient dNdmu with shape {H.shape} to mu length {len(mu)}")

def plot_mu_heatmap_and_asym(path, label, t_range_fs=None, vmin_mode="percentile", p_lo=0.5, p_hi=99.5):
    axes = load_axes(path)
    sp = load_spectra(path)
    t = np.asarray(axes["time_fs"])
    mu = np.asarray(axes["mu_mid"])
    dNdmu = sp["He_e_dNdmu"]

    # --- NEW: time window mask ---
    if t_range_fs is not None:
        tmin, tmax = t_range_fs
        mask_t = (t >= tmin) & (t <= tmax)
        if not np.any(mask_t):
            raise RuntimeError(f"No points in requested t_range_fs={t_range_fs}.")
        t = t[mask_t]
        dNdmu = np.asarray(dNdmu)[mask_t, :] if np.asarray(dNdmu).shape[0] == len(axes["time_fs"]) else np.asarray(dNdmu)[:, mask_t]
    # -----------------------------

    Hmu_t = _orient_mu_t(dNdmu, mu)   # (Nmu, Nt) AFTER slicing
    Ht_mu = Hmu_t.T
    A = asymmetry_from_dNdmu(mu, Ht_mu)

    Hplot = np.array(Hmu_t, dtype=float)
    Hplot[Hplot <= 0] = np.nan

    finite = Hplot[np.isfinite(Hplot)]
    if finite.size == 0:
        raise RuntimeError("All heatmap values are zero/NaN after masking.")

    if vmin_mode == "percentile":
        vmin = np.percentile(finite, p_lo)
        vmax = np.percentile(finite, p_hi)
    else:
        vmin = np.nanmin(finite)
        vmax = np.nanmax(finite)

    vmin = max(vmin, 1e-1)
    vmax = max(vmax, vmin * 10)

    fig = plt.figure(figsize=(7.2, 3.4), constrained_layout=False)
    gs = fig.add_gridspec(2, 2, width_ratios=[1.0, 0.05], height_ratios=[3, 1], wspace=0.01, hspace=0.08)

    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[1, 0], sharex=ax0)
    cax = fig.add_subplot(gs[0, 1])

    im = ax0.imshow(
        Hplot,
        origin="lower",
        aspect="auto",
        extent=[t.min(), t.max(), mu.min(), mu.max()],
        cmap="viridis",
        norm=LogNorm(vmin=vmin, vmax=vmax),
        #interpolation="nearest",
    )
    cb = fig.colorbar(im, cax=cax)
    cb.set_label(r"$dN/d\mu$ (arb.)")

    ax0.set_ylabel(r"$\mu = \cos\theta$")
    ax0.set_title(f"{label}: electron angular distribution vs time")
    plt.setp(ax0.get_xticklabels(), visible=False)

    ax1.plot(t, A, lw=2.0)
    ax1.axhline(0, lw=1.0)
    ax1.set_ylabel("A(t)")
    ax1.set_xlabel("time (fs)")
    ax1.grid(True, alpha=0.25)

    plt.show()

# Run
#plot_mu_heatmap_and_asym(H5_IONIZED, "Pre-ionized", t_range_fs=(-80, 40))
plot_mu_heatmap_and_asym(H5_NEUTRAL, "Neutral", t_range_fs=(-20, 25))


# Figure 4 — Spectral fingerprints (heatmaps)

From `/spectra` + `/axes` you can do:
- Electron spectra heatmap: `He_e_dNdE(t,E)`
- Ion spectra heatmap: `He_i_dNdE(t,E)`
- Optional: overlay a few lineouts at key times


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

def _nearest_index(x, x0):
    x = np.asarray(x)
    return int(np.argmin(np.abs(x - x0)))

def hist_quantile(E_mid, counts, q=0.5):
    h = np.asarray(counts, float)
    h[~np.isfinite(h)] = 0.0
    tot = h.sum()
    if tot <= 0:
        return np.nan
    cdf = np.cumsum(h) / tot
    j = int(np.searchsorted(cdf, q, side="left"))
    j = min(max(j, 0), len(E_mid)-1)
    return float(E_mid[j])

def plot_fig3_lineouts_with_insets(
    h5,
    times_fs=(-25, -20, -15, -10, -5, 0, 10, 25),
    E_range_eV=(0.1, 400),
    xscale="linear",
    yscale="log",
    legend_ncol=1,
    q_med=0.5,
):
    axes = h5["axes"]
    spec = h5["spectra"]

    t = np.asarray(axes["time_fs"][:], float)
    E = np.asarray(axes["E_mid_eV"][:], float)

    He = np.asarray(spec["He_e_dNdE"][:], float)
    Hi = np.asarray(spec["He_i_dNdE"][:], float)

    # orient to (Nt, NE)
    if He.shape[0] != len(t) and He.shape[1] == len(t):
        He = He.T
    if Hi.shape[0] != len(t) and Hi.shape[1] == len(t):
        Hi = Hi.T

    # map requested times -> indices, de-dup
    idxs_raw = [_nearest_index(t, tt) for tt in times_fs]
    idxs = []
    for i in idxs_raw:
        if i not in idxs:
            idxs.append(i)

    # figure with two rows, shared x
    fig, (ax_e, ax_i) = plt.subplots(
        2, 1, figsize=(5,7), sharex=True,
        gridspec_kw={"hspace": 0.0, "height_ratios": [1, 1]}
    )

    # ---- TOP: electrons ----
    for i in idxs:
        ax_e.plot(E, He[i], lw=2.0, label=f"t={t[i]:.1f} fs")
    ax_e.set_ylabel("e⁻ counts/bin ")
    ax_e.set_title("Fig. 3 — Kinetic energy spectra at selected times")
    #ax_e.grid(False, which="both", alpha=0.25)

    # ---- BOTTOM: ions ----
    for i in idxs:
        ax_i.plot(E, Hi[i], lw=2.0, label=f"t={t[i]:.1f} fs")
    ax_i.set_ylabel("ion counts/bin ")
    ax_i.set_xlabel("E (eV)")
    #ax_i.grid(False, which="both", alpha=0.25)

    # scales
    if xscale == "log":
        ax_i.set_xscale("log")
    if yscale == "log":
        ax_e.set_yscale("log")
        ax_i.set_yscale("log")

    if E_range_eV is not None:
        ax_i.set_xlim(E_range_eV)

    # legends (avoid covering the inset by placing them left)
    ax_i.legend(loc="upper left", ncol=legend_ncol, fontsize=10, frameon=False)
    #ax_i.legend(loc="upper right", ncol=legend_ncol, fontsize=10, frameon=False)

    # ---- Insets: median energy vs time (one per subplot) ----
    t_sel = np.array([t[i] for i in idxs], float)
    order = np.argsort(t_sel)
    t_sel = t_sel[order]

    Em_e = np.array([hist_quantile(E, He[i], q=q_med) for i in idxs], float)[order]
    Em_i = np.array([hist_quantile(E, Hi[i], q=q_med) for i in idxs], float)[order]

    # Electron inset (top axis)
    axins_e = inset_axes(ax_e, width="40%", height="40%", loc="upper right", borderpad=2.0)
    axins_e.plot(t_sel, Em_e, ".--",color='gray')
    axins_e.set_xlabel("t (fs)", fontsize=9)
    axins_e.set_ylabel(r"$KE$ (eV)", fontsize=9)
    axins_e.tick_params(labelsize=8)
    axins_e.grid(True, alpha=0.25)
    axins_e.set_title("e⁻ median", fontsize=9)

    # Ion inset (bottom axis)
    axins_i = inset_axes(ax_i, width="40%", height="40%", loc="upper right", borderpad=2.0)
    axins_i.plot(t_sel, Em_i, ".--",color='gray')
    axins_i.set_xlabel("t (fs)", fontsize=9)
    axins_i.set_ylabel(r"$KE$ (eV)", fontsize=9)
    axins_i.tick_params(labelsize=8)
    axins_i.grid(True, alpha=0.25)
    axins_i.set_title("ion median", fontsize=9)

    plt.tight_layout()
    plt.show()
    plt.close(fig)   # prevents duplicate display in notebooks


In [ ]:
plot_fig3_lineouts_with_insets(
    h5,
    times_fs=(-15, 0, 10, 25,50,80,120),  E_range_eV=(0.1, 2000),
)


In [ ]:
if H5_NEUTRAL_LOG is not None:
    plot_combined_spectrum(H5_NEUTRAL_LOG, "Pre-ionized/log-reduced", t_range_fs=(-11, 125))
else:
    print("H5_NEUTRAL_LOG not found. Skipping log-reduced spectrum plot.")


# Figure 5 — Experiment comparison (VMI-style)

This is only possible **if** your reduced H5 contains 2D momentum histograms, e.g.:
- `spectra/He_e_H_pxy` and axes `axes/He_e_p_mid_SI`

If present, this cell will plot a single VMI image at a chosen time index.


In [ ]:
def plot_vmi_from_h5(path, label, species="He_e", time_index=-1):
    # optional data: spectra/{species}_H_pxy with axes/{species}_p_mid_SI
    H_key = f"spectra/{species}_H_pxy"
    p_mid_key = f"axes/{species}_p_mid_SI"

    if not h5_has(path, H_key) or not h5_has(path, p_mid_key):
        print(f"[{label}] No VMI hist found ({H_key}). Skipping.")
        return

    H = h5_read(path, H_key)        # expected (Nt, Np, Np)
    p = h5_read(path, p_mid_key)    # (Np,)

    if H.ndim != 3:
        print(f"[{label}] Unexpected shape for {H_key}: {H.shape}")
        return

    axes_ = load_axes(path)
    t = axes_["time_fs"]
    i = time_index if time_index >= 0 else (H.shape[0] + time_index)

    img = H[i]
    plt.figure(figsize=(6,6))
    plt.imshow(img, origin="lower", aspect="equal",
               extent=[p.min()/2, p.max()/2, p.min()/2, p.max()/2])
    plt.xlabel("p_x (SI)")
    plt.ylabel("p_y (SI)")
    plt.title(f"{label}: {species} VMI (t={t[i]:.0f} fs)")
    plt.colorbar(label="counts (arb.)")
    plt.tight_layout()
    plt.show()

plot_vmi_from_h5(H5_NEUTRAL, "Neutral", "He_e", time_index=-1)
if H5_IONIZED is not None:
    plot_vmi_from_h5(H5_IONIZED, "Pre-ionized", "He_e", time_index=-1)
else:
    print("H5_IONIZED not set. Skipping pre-ionized VMI plot.")


## Plotting parameters as SCIPY

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- Try SciencePlots first (preferred) ---
try:
    import scienceplots  # required before plt.style.use('science') in newer versions
    plt.style.use(["science", "grid", "no-latex"])
    # You can also try: ["science", "nature", "no-latex"] depending on what’s available in your install.
    USING = "SciencePlots"
except Exception as e:
    USING = f"rcParams fallback (SciencePlots not available: {type(e).__name__})"
    import matplotlib.colors as mcolors
    
    def muted_tab10(gray=0.55, mix=0.35):
        """
        gray: 0..1 gray level (0=black, 1=white)
        mix:  0..1 how much gray to mix in (0=original, 1=all gray)
        """
        base = plt.cm.tab10.colors
        g = (gray, gray, gray)
        return [tuple((1-mix)*c[i] + mix*g[i] for i in range(3)) for c in base]
    
    plt.rcParams["axes.prop_cycle"] = plt.cycler(color=muted_tab10(gray=0.55, mix=0.10))
    mpl.rcParams.update({
        # Figure / saving
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,

        # Fonts (LaTeX-like look without requiring LaTeX)
        "font.family": "serif",
        "font.size": 16,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 12,

        # Axes
        "axes.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,

        # Grid (light, paper-style)
        "grid.alpha": 0.25,
        "grid.linewidth": 0.6,
        "grid.linestyle": "-",

        # Lines
        "lines.linewidth": 2.0,
        "lines.markersize": 5,

        # Ticks
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 4,
        "ytick.major.size": 4,
        "xtick.minor.size": 2.5,
        "ytick.minor.size": 2.5,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.minor.width": 0.6,
        "ytick.minor.width": 0.6,
        "xtick.minor.visible": False,
        "ytick.minor.visible": False,

        # Legend
        "legend.frameon": False,
    })

print("Plot style:", USING)
mpl.rcParams.update({
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.spines.left": True,
    "axes.spines.bottom": True,
    "axes.linewidth": 0.9,
})



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py

def plot_net_cluster_charge(path, title="Net cluster charge vs time", Zmax=2):
    """
    Net charge:
        Q_net(t) = Q_ions(t) - N_e(t)

    Convention:
      electrons = -1
      ions      = +Z

    Preferred fast path from reduced H5:
      timeseries/He_e_N_real
      timeseries/He_i_charge_frac
      timeseries/He_i_N_real   (or He_i_N_macro as fallback)

    Fallback:
      read step-resolved particle data from /macrocell3d/step_xxxxxxxx if present.
    """

    axes = load_axes(path)
    ts = load_timeseries(path)
    t = np.asarray(axes["time_fs"], float)

    # ----------------------------
    # 1) electrons
    # ----------------------------
    if "He_e_N_real" in ts:
        Ne = np.asarray(ts["He_e_N_real"], float)
    elif "He_e_N_macro" in ts:
        Ne = np.asarray(ts["He_e_N_macro"], float)
    else:
        raise RuntimeError("Could not find He_e_N_real or He_e_N_macro in timeseries.")

    # ----------------------------
    # 2) ions: total positive charge
    # ----------------------------
    Qi = None

    # Fast path: from charge fractions
    if "He_i_charge_frac" in ts:
        fracZ = np.asarray(ts["He_i_charge_frac"], float)

        # orient to (Nt, Zmax+1)
        if fracZ.ndim != 2:
            raise RuntimeError(f"Unexpected shape for He_i_charge_frac: {fracZ.shape}")

        if fracZ.shape[0] != len(t) and fracZ.shape[1] == len(t):
            fracZ = fracZ.T

        if fracZ.shape[0] != len(t):
            raise RuntimeError(
                f"He_i_charge_frac shape {fracZ.shape} incompatible with len(time_fs)={len(t)}"
            )

        if "He_i_N_real" in ts:
            Ni = np.asarray(ts["He_i_N_real"], float)
        elif "He_i_N_macro" in ts:
            Ni = np.asarray(ts["He_i_N_macro"], float)
        else:
            raise RuntimeError("Could not find He_i_N_real or He_i_N_macro in timeseries.")

        Zvals = np.arange(fracZ.shape[1], dtype=float)   # [0,1,2,...]
        meanZ = np.sum(fracZ * Zvals[None, :], axis=1)
        Qi = Ni * meanZ

    # Fallback: stepwise read from macrocell3d
    elif h5_has(path, "macrocell3d"):
        print("[info] He_i_charge_frac not found, using /macrocell3d fallback...")
        with h5py.File(path, "r") as f:
            step_keys = sorted(list(f["macrocell3d"].keys()))

            Qi_list = []
            t_list = []

            for sk in step_keys:
                grp = f["macrocell3d"][sk]

                # time from step name if needed
                step = int(sk.split("_", 1)[1])

                # try stored time axis mapping first
                if len(step_keys) == len(t):
                    t_list.append(t[len(t_list)])
                else:
                    t_list.append(step)

                if "He_i_boundElectrons_hist" in grp and "He_i_weight_sum" in grp:
                    # example fallback if histograms were saved
                    be_hist = np.array(grp["He_i_boundElectrons_hist"], float)
                    Ni_step = float(np.array(grp["He_i_weight_sum"]))

                    # boundElectrons 2,1,0  -> charge Z = 0,1,2 for helium
                    # adjust if your saved histogram uses another ordering
                    be_vals = np.arange(len(be_hist))
                    Zvals = Zmax - be_vals
                    Qi_list.append(np.sum(be_hist * Zvals))
                else:
                    Qi_list.append(np.nan)

        Qi = np.asarray(Qi_list, float)
        t = np.asarray(t_list, float)

    else:
        raise RuntimeError(
            "Could not compute ion charge: neither timeseries/He_i_charge_frac "
            "nor suitable macrocell3d fallback was found."
        )

    # ----------------------------
    # 3) net charge
    # ----------------------------
    Qnet = Qi - Ne

    # sort just in case
    order = np.argsort(t)
    t = t[order]
    Ne = Ne[order] if len(Ne) == len(order) else Ne
    Qi = Qi[order]
    Qnet = Qnet[order]

    # ----------------------------
    # 4) plot
    # ----------------------------
    plt.figure(figsize=(8, 4.5))
    plt.plot(t, Qnet, "o-", lw=2, ms=4, label=r"$Q_{\mathrm{net}} = Q_i - N_e$")
    plt.axhline(0, color="k", lw=1)

    plt.xlabel("time (fs) (0 = laser peak)")
    plt.ylabel("net charge (real-particle units)")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return t, Qnet, Qi, Ne


In [ ]:
t, Qnet, Qi, Ne = plot_net_cluster_charge(
    H5_NEUTRAL,
    title="Run004: total net cluster charge vs time"
)


## Optional experiment / raw-openPMD comparison cells

The cells below are restored from the old paper notebook. They require optional files such as `all_runs_ion_kinematics.h5`, `all_runs_vmi_radii.h5`, and access to the raw `SIM_OUTPUT` BP5 files. If those paths are not available, skip those cells or edit the top path configuration.


In [ ]:
# =========================================================
# FIG. 5 ion panel:
# Ion momentum distributions (simulation vs experiment)
# momentum in atomic units
# =========================================================

from nanoplasma_analysis import NanoPlasmaRun
import scipy.constants as sc
import numpy as np
import matplotlib.pyplot as plt
import h5py

# -----------------------
# Load simulation
# -----------------------
run = NanoPlasmaRun(
    path=str(SIM_OUTPUT),
    laser_peak_at_target=89603
)

# -----------------------
# User parameters
# -----------------------
ION_SPECIES = "He_i"

# simulation times to show
TIMES_FS = (0, 120)

# experimental run to compare
EXP_H5 = EXP_ION_H5
RUN_EXP = 236
EXP_LABEL = "Exp. 16 fs"

NBINS = 1000
P_MAX_AU = None
XMAX = 500
LOGY = True

# atomic unit of momentum [kg m/s]
P_AU_SI = sc.physical_constants["atomic unit of momentum"][0]

# -----------------------
# Helpers
# -----------------------
def extract_step_from_filename(filename):
    return int((filename.rpartition('_')[2]).rpartition('.')[0])

# time axis from raw files
steps = np.array([extract_step_from_filename(fn) for fn in run.files], dtype=int)
times_all_fs = np.array([run.time_fs_from_step(st) for st in steps], dtype=float)

# nearest saved outputs to requested times
sel_idx = []
for tt in TIMES_FS:
    i = int(np.argmin(np.abs(times_all_fs - tt)))
    if i not in sel_idx:
        sel_idx.append(i)

print("Requested times [fs]:", TIMES_FS)
print("Selected saved times [fs]:", [f"{times_all_fs[i]:.2f}" for i in sel_idx])

# -----------------------
# Determine common momentum range from simulation
# -----------------------
pmax_found = 0.0

for i in sel_idx:
    fn = run.files[i]
    step = steps[i]

    px, py, pz, w = run._read_momentum_and_weight(fn, step, ION_SPECIES)
    if px is None:
        print(f"Skipping step={step}: no {ION_SPECIES}")
        continue

    p_abs_au = np.sqrt(px*px + py*py + pz*pz) / P_AU_SI
    if p_abs_au.size > 0:
        pmax_found = float(np.maximum(pmax_found, np.nanmax(p_abs_au)))

if P_MAX_AU is None:
    P_MAX_AU = 1.05 * pmax_found if pmax_found > 0 else 1.0

edges = np.linspace(0.0, P_MAX_AU, NBINS + 1)
mid = 0.5 * (edges[1:] + edges[:-1])

# -----------------------
# Plot
# -----------------------
fig, ax = plt.subplots(figsize=(7.0, 3))

# simulation curves
for i in sel_idx:
    fn = run.files[i]
    step = steps[i]
    t_fs = times_all_fs[i]

    px, py, pz, w = run._read_momentum_and_weight(fn, step, ION_SPECIES)
    if px is None:
        continue

    # true 3D momentum magnitude
    p_abs_au = np.sqrt(px*px + py*py + pz*pz) / P_AU_SI

    hist, _ = np.histogram(p_abs_au, bins=edges, weights=w)

    if np.nanmax(hist) > 0:
        hist = hist / np.nanmax(hist)

    ax.plot(mid, hist, lw=2.2, label=f"Sim. {t_fs:.0f} fs")

# experimental curve
with h5py.File(EXP_H5, "r") as h5:
    c_exp = h5[f"runs/run_{RUN_EXP}/hist/counts_P"][:].astype(float)
    p_exp = h5[f"runs/run_{RUN_EXP}/hist/centers_P"][:]

if np.nanmax(c_exp) > 0:
    c_exp = c_exp / np.nanmax(c_exp)

ax.plot(
    p_exp-60, c_exp,
    "o-", ms=4, lw=1.8,
    label=EXP_LABEL
)

# styling
ax.set_xlabel(r"$|p|$ (a.u.)")
ax.set_ylabel("Normalized counts")
ax.set_xlim(0, 300)
ax.set_ylim(0.01, 1+0.1)
if LOGY:
    ax.set_yscale("log")
ax.grid(True, alpha=0.25)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
# =========================================================
# FIG. 5 ion panel:
# Combined ion momentum distribution from simulation
# (0 fs convolved with ~100 fs), aligned to experimental peak
# momentum in atomic units
# =========================================================

from nanoplasma_analysis import NanoPlasmaRun
import scipy.constants as sc
import numpy as np
import matplotlib.pyplot as plt
import h5py

# -----------------------
# Load simulation
# -----------------------
run = NanoPlasmaRun(
    path=str(SIM_OUTPUT),
    laser_peak_at_target=89603
)

# -----------------------
# User parameters
# -----------------------
ION_SPECIES = "He_i"

# two simulation times to combine
TIME_A_FS = 0
TIME_B_FS = 100

# experimental run to compare
EXP_H5 = EXP_ION_H5
RUN_EXP = 236
EXP_LABEL = "Exp."

NBINS = 1000
P_MAX_AU = None
XMAX = 300
LOGY = True

# choose combination mode:
#   "convolution"  -> combined = h0 (*) h100
#   "product"      -> combined = h0 * h100
COMBINE_MODE = "product"

# keep your experimental horizontal correction if needed
EXP_X_SHIFT = -60.0

# atomic unit of momentum [kg m/s]
P_AU_SI = sc.physical_constants["atomic unit of momentum"][0]

# -----------------------
# Helpers
# -----------------------
def extract_step_from_filename(filename):
    return int((filename.rpartition('_')[2]).rpartition('.')[0])

def get_hist_for_time(run, species, target_time_fs, edges):
    """Return nearest saved time and normalized histogram dN/dp on 'edges'."""
    steps = np.array([extract_step_from_filename(fn) for fn in run.files], dtype=int)
    times_all_fs = np.array([run.time_fs_from_step(st) for st in steps], dtype=float)

    i = int(np.argmin(np.abs(times_all_fs - target_time_fs)))
    fn = run.files[i]
    step = steps[i]
    t_fs = times_all_fs[i]

    px, py, pz, w = run._read_momentum_and_weight(fn, step, species)
    if px is None:
        raise RuntimeError(f"No particles found for {species} at nearest step to {target_time_fs} fs.")

    p_abs_au = np.sqrt(px*px + py*py + pz*pz) / P_AU_SI
    hist, _ = np.histogram(p_abs_au, bins=edges, weights=w)
    hist = hist.astype(float)

    # normalize to peak = 1
   # if np.nanmax(hist) > 0:
      #  hist /= np.nanmax(hist)

    return t_fs, hist

# -----------------------
# Time axis from raw files
# -----------------------
steps = np.array([extract_step_from_filename(fn) for fn in run.files], dtype=int)
times_all_fs = np.array([run.time_fs_from_step(st) for st in steps], dtype=float)

# nearest saved outputs
i0 = int(np.argmin(np.abs(times_all_fs - TIME_A_FS)))
i1 = int(np.argmin(np.abs(times_all_fs - TIME_B_FS)))

print("Requested times [fs]:", (TIME_A_FS, TIME_B_FS))
print("Selected saved times [fs]:", f"{times_all_fs[i0]:.2f}", f"{times_all_fs[i1]:.2f}")

# -----------------------
# Determine common momentum range from the two simulation times
# -----------------------
pmax_found = 0.0
for i in [i0, i1]:
    fn = run.files[i]
    step = steps[i]

    px, py, pz, w = run._read_momentum_and_weight(fn, step, ION_SPECIES)
    if px is None:
        continue

    p_abs_au = np.sqrt(px*px + py*py + pz*pz) / P_AU_SI
    if p_abs_au.size > 0:
        pmax_found = max(pmax_found, float(np.nanmax(p_abs_au)))

if P_MAX_AU is None:
    P_MAX_AU = 1.05 * pmax_found if pmax_found > 0 else 1.0

edges = np.linspace(0.0, P_MAX_AU, NBINS + 1)
mid = 0.5 * (edges[1:] + edges[:-1])
dp = edges[1] - edges[0]

# -----------------------
# Build the two simulation histograms
# -----------------------
t0_fs, h0 = get_hist_for_time(run, ION_SPECIES, TIME_A_FS, edges)
t1_fs, h1 = get_hist_for_time(run, ION_SPECIES, TIME_B_FS, edges)

# -----------------------
# Combine them
# -----------------------
if COMBINE_MODE.lower() == "convolution":
    # discrete convolution; multiply by dp to better mimic continuous convolution
    h_comb = np.convolve(h0, h1, mode="full") * dp
    p_comb = np.arange(h_comb.size) * dp
elif COMBINE_MODE.lower() == "product":
    h_comb = h0 + h1
    p_comb = mid.copy()
else:
    raise ValueError("COMBINE_MODE must be 'convolution' or 'product'.")

# normalize combined curve to peak = 1
if np.nanmax(h_comb) > 0:
    h1 /= np.nanmax(h0)
    h_comb = h_comb / np.nanmax(h0)
    h0 /= np.nanmax(h0)


# -----------------------
# Load experiment
# -----------------------
with h5py.File(EXP_H5, "r") as h5:
    c_exp = h5[f"runs/run_{RUN_EXP}/hist/counts_P"][:].astype(float)
    p_exp = h5[f"runs/run_{RUN_EXP}/hist/centers_P"][:].astype(float)

if np.nanmax(c_exp) > 0:
    c_exp = c_exp / np.nanmax(c_exp)

# experimental x as displayed
p_exp_plot = p_exp 
# -----------------------
# Align combined simulation peak to experimental peak
# -----------------------
p_peak_exp = p_exp_plot[np.nanargmax(c_exp)]
p_peak_comb = p_comb[np.nanargmax(h_comb)]
shift_comb = p_peak_exp - p_peak_comb
p_comb_aligned = p_comb + shift_comb

print(f"Experimental peak at: {p_peak_exp:.3f} a.u.")
print(f"Combined sim peak before shift: {p_peak_comb:.3f} a.u.")
print(f"Applied horizontal shift to combined sim: {shift_comb:.3f} a.u.")

# -----------------------
# Plot
# -----------------------
fig, ax = plt.subplots(figsize=(12.0, 3.0))

# optional: plot the two ingredients lightly
ax.plot(mid, h0, lw=1.2, alpha=0.8, ls="--", label=f"Sim. t={t0_fs:.0f} fs ")
ax.plot(mid, h1, lw=1.2, alpha=0.8, ls=":",  label=f"Sim. t={t1_fs:.0f} fs ")

# combined curve
ax.plot(p_comb_aligned, h_comb, lw=2.4, label=rf"$\mathrm{{t}}_{{{t0_fs:.0f}\,\mathrm{{fs}}}} + \mathrm{{t}}_{{{t1_fs:.0f}\,\mathrm{{fs}}}}$ (shifted)" )

# experimental curve
ax.plot(
    p_exp_plot, c_exp,
    ".-", ms=4, lw=1.8,
    label=EXP_LABEL
)

# styling
ax.set_xlabel(r"$|p|$ (a.u.)")
ax.set_ylabel("Normalized counts")
ax.set_xlim(0, XMAX)
ax.set_ylim(0.01 if LOGY else 0.0, 1.1)
if LOGY:
    ax.set_yscale("log")
ax.grid(True, alpha=0.25)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
# =========================================================
# FIGURE:
# top    = Sim / Exp VMI
# middle = mean single-shot max electron KE vs IR (experimental)
# bottom = ion momentum distribution: summed sim (t0 + t106) vs experiment
# =========================================================

import h5py
import numpy as np
import matplotlib.pyplot as plt
import scipy.constants as sc
import matplotlib as mpl

from nanoplasma_analysis import NanoPlasmaRun

# -----------------------
# Reset plot style
# -----------------------
plt.style.use("default")
mpl.rcdefaults()
mpl.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "axes.edgecolor": "black",
    "axes.linewidth": 1.0,
    "axes.grid": False,
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "legend.frameon": True,
    "image.cmap": "inferno",
})

# -----------------------
# Inputs
# -----------------------
reduced_h5_sim = H5_NEUTRAL
exp_vmi_h5 = exp_vmi_h5

# raw sim for ion momentum
run = NanoPlasmaRun(
    path=str(SIM_OUTPUT),
    laser_peak_at_target=89603
)

RUN_TARGET_VMI = 325
TIME_INDEX = -1

XC_MANUAL, YC_MANUAL = 119.01, 109.16
PX2_TO_EV = 3.9914e-3
PAD_FACTOR = 2.0

# ion panel inputs
ION_SPECIES = "He_i"
SIM_SUM_TIMES_FS = (0, 106)   # summed raw histograms before normalization
EXP_ION_H5 = EXP_ION_H5
RUN_EXP_ION = 236
EXP_ION_LABEL = "Experiment"
ION_NBINS = 1000
ION_XMAX = 500
SHOW_COMPONENTS = True   # show faint t0 and t106 ingredients
LOGY = False

EV_PER_HA = sc.physical_constants["Hartree energy in eV"][0]
p_au_SI = sc.physical_constants["atomic unit of momentum"][0]

# -----------------------
# Helpers
# -----------------------
def extract_step_from_filename(filename):
    return int((filename.rpartition('_')[2]).rpartition('.')[0])
def sim_vmi_edge_energy_eV(h5_path, time_index, q=0.97):
    """
    Compute a max-like characteristic energy from a simulated VMI image
    using the q-quantile of the radial momentum distribution.

    Returns
    -------
    E_q_eV : float
        Quantile energy in eV
    p_q_au : float
        Quantile momentum in a.u.
    """
    with h5py.File(h5_path, "r") as h5:
        H = h5["spectra/He_e_H_pxy"][time_index, :, :].astype(float)
        p_mid_SI = h5["axes/He_e_p_mid_SI"][:]

    p_mid_au = p_mid_SI / p_au_SI
    PX, PY = np.meshgrid(p_mid_au, p_mid_au, indexing="xy")
    Pabs = np.sqrt(PX**2 + PY**2)

    W = np.clip(H, 0.0, None).ravel()
    P = Pabs.ravel()

    m = np.isfinite(P) & np.isfinite(W) & (W > 0)
    if not np.any(m):
        return np.nan, np.nan

    P = P[m]
    W = W[m]

    order = np.argsort(P)
    P = P[order]
    W = W[order]

    cdf = np.cumsum(W)
    cdf = cdf / cdf[-1]

    p_q_au = np.interp(q, cdf, P)
    E_q_eV = 0.5 * (p_q_au ** 2) * EV_PER_HA
    return E_q_eV, p_q_au


def sim_vmi_mean_energy_eV(h5_path, time_index):
    """
    Intensity-weighted mean electron energy from a simulated VMI image.
    Useful as an alternative to the edge-based observable.
    """
    with h5py.File(h5_path, "r") as h5:
        H = h5["spectra/He_e_H_pxy"][time_index, :, :].astype(float)
        p_mid_SI = h5["axes/He_e_p_mid_SI"][:]

    p_mid_au = p_mid_SI / p_au_SI
    PX, PY = np.meshgrid(p_mid_au, p_mid_au, indexing="xy")
    E_map_eV = 0.5 * (PX**2 + PY**2) * EV_PER_HA

    W = np.clip(H, 0.0, None)
    denom = np.nansum(W)
    if denom <= 0:
        return np.nan

    return np.nansum(W * E_map_eV) / denom
def get_hist_for_time(run, species, target_time_fs, edges, p_au_SI):
    """
    Return nearest saved time and RAW histogram dN/dp on 'edges'.
    No normalization here.
    """
    steps = np.array([extract_step_from_filename(fn) for fn in run.files], dtype=int)
    times_all_fs = np.array([run.time_fs_from_step(st) for st in steps], dtype=float)

    i = int(np.argmin(np.abs(times_all_fs - target_time_fs)))
    fn = run.files[i]
    step = steps[i]
    t_fs = times_all_fs[i]

    px, py, pz, w = run._read_momentum_and_weight(fn, step, species)
    if px is None:
        raise RuntimeError(f"No particles found for {species} near {target_time_fs} fs.")

    p_abs_au = np.sqrt(px*px + py*py + pz*pz) / p_au_SI
    hist, _ = np.histogram(p_abs_au, bins=edges, weights=w)
    return t_fs, hist.astype(float)

def sci_label(val):
    exp = int(np.floor(np.log10(val)))
    coeff = val / 10**exp
    if abs(coeff - round(coeff)) < 1e-12:
        coeff = int(round(coeff))
    return rf"${coeff} \times 10^{{{exp}}}$"

# -----------------------
# Simulation VMI
# -----------------------
with h5py.File(reduced_h5_sim, "r") as h5:
    H_pxy_e = h5["spectra/He_e_H_pxy"][TIME_INDEX, :, :]
    p_mid_sim_SI = h5["axes/He_e_p_mid_SI"][:]

p_mid_sim_au = p_mid_sim_SI / p_au_SI
dp_sim_au = p_mid_sim_au[1] - p_mid_sim_au[0]
sim_extent_au = [
    p_mid_sim_au[0] - 0.5 * dp_sim_au,
    p_mid_sim_au[-1] + 0.5 * dp_sim_au,
    p_mid_sim_au[0] - 0.5 * dp_sim_au,
    p_mid_sim_au[-1] + 0.5 * dp_sim_au,
]

# -----------------------
# Experimental VMI for top-right
# -----------------------
with h5py.File(exp_vmi_h5, "r") as h5:
    run_path = f"runs/run_{RUN_TARGET_VMI}"
    if run_path not in h5:
        raise ValueError(f"{run_path} not found in experimental H5")
    H_sum_vmi = h5[f"{run_path}/summed/image"][:].astype(float)

ny, nx = H_sum_vmi.shape
PX2_TO_P_AU = np.sqrt(2.0 * PX2_TO_EV / EV_PER_HA)

exp_extent_au = [
    (0.0      - XC_MANUAL) * PX2_TO_P_AU,
    (nx - 1.0 - XC_MANUAL) * PX2_TO_P_AU,
    (0.0      - YC_MANUAL) * PX2_TO_P_AU,
    (ny - 1.0 - YC_MANUAL) * PX2_TO_P_AU,
]

# zero-pad experimental image
ny_pad = int(np.ceil(ny * PAD_FACTOR))
nx_pad = int(np.ceil(nx * PAD_FACTOR))
if ny_pad % 2 != ny % 2:
    ny_pad += 1
if nx_pad % 2 != nx % 2:
    nx_pad += 1

H_exp_pad = np.zeros((ny_pad, nx_pad), dtype=float)
y0 = int((ny_pad - ny) // 2)
x0 = int((nx_pad - nx) // 2)
H_exp_pad[y0:y0+ny, x0:x0+nx] = H_sum_vmi

# display range for both VMI panels
p_plot_au = max(
    abs(sim_extent_au[0]), abs(sim_extent_au[1]),
    abs(sim_extent_au[2]), abs(sim_extent_au[3]),
    abs(exp_extent_au[0]), abs(exp_extent_au[1]),
    abs(exp_extent_au[2]), abs(exp_extent_au[3]),
)
# -----------------------
# Collect run summary for middle panel
# -----------------------
summary = []
with h5py.File(exp_vmi_h5, "r") as h5:
    for run_name in sorted(h5["runs"].keys(), key=lambda s: int(s.split("_")[1])):
        grp = h5[f"runs/{run_name}"]

        run_no = int(grp.attrs["run_no"]) if "run_no" in grp.attrs else int(run_name.split("_")[1])
        ir = grp.attrs.get("ir_mj", np.nan)
        temp = grp.attrs.get("temp_K", np.nan)

        if "shots" not in grp:
            continue
        if "radius_px" not in grp["shots"] or "energy_eV" not in grp["shots"]:
            continue

        r_shots = grp["shots"]["radius_px"][:]
        e_shots = grp["shots"]["energy_eV"][:]

        mr = np.isfinite(r_shots) & (r_shots > 0)
        me = np.isfinite(e_shots) & (e_shots > 0)

        if np.any(mr):
            r_mean = np.mean(r_shots[mr])
            r_std  = np.std(r_shots[mr])
        else:
            r_mean = np.nan
            r_std  = np.nan

        if np.any(me):
            e_mean = np.mean(e_shots[me])
            e_std  = np.std(e_shots[me])
        else:
            e_mean = np.nan
            e_std  = np.nan

        summary.append({
            "run": run_no,
            "ir": float(ir) if np.isfinite(ir) else np.nan,
            "temp": float(temp) if np.isfinite(temp) else np.nan,
            "r_mean": r_mean,
            "r_std": r_std,
            "e_mean": e_mean,
            "e_std": e_std,
        })

temps = sorted(set(s["temp"] for s in summary if np.isfinite(s["temp"])))

# -----------------------
# Figure layout
# -----------------------
fig = plt.figure(figsize=(12.8, 13.2))

outer = fig.add_gridspec(
    3, 1,
    height_ratios=[1.15, 1.15, 0.85],
    hspace=0.30
)

top = outer[0].subgridspec(
    1, 4,
    width_ratios=[1.0, 0.035, 1.0, 0.035],
    wspace=0
)

#ax_sim  = fig.add_subplot(top[0, 0])
#cax_sim = fig.add_subplot(top[0, 1])
#ax_exp  = fig.add_subplot(top[0, 2], sharex=ax_sim, sharey=ax_sim)
#cax_exp = fig.add_subplot(top[0, 3])

ax_mid = fig.add_subplot(outer[1, 0])
ax_bot = fig.add_subplot(outer[2, 0])

# -----------------------
# Top row: VMI panels
# -----------------------
sim_vmax = np.nanpercentile(H_pxy_e, 99) if np.any(np.isfinite(H_pxy_e)) else 1.0
exp_vmax = np.nanpercentile(H_exp_pad, 99) if np.any(np.isfinite(H_exp_pad)) else 1.0

# axes (no sharing if you want independent limits)
ax_sim  = fig.add_subplot(top[0, 0])
cax_sim = fig.add_subplot(top[0, 1])
ax_exp  = fig.add_subplot(top[0, 2])
cax_exp = fig.add_subplot(top[0, 3])

# sim image
im0 = ax_sim.imshow(
    H_pxy_e,
    origin="lower",
    cmap="inferno",
    extent=sim_extent_au,
    aspect="equal",
    vmin=0.0,
    vmax=sim_vmax,
)

# experimental padded image with correct padded extent
XC_PAD = XC_MANUAL + x0
YC_PAD = YC_MANUAL + y0
PX2_TO_P_AU = np.sqrt(2.0 * PX2_TO_EV / EV_PER_HA)

exp_extent_au = [
    (0.0      - XC_MANUAL) * PX2_TO_P_AU,
    (nx - 1.0 - XC_MANUAL) * PX2_TO_P_AU,
    (0.0      - YC_MANUAL) * PX2_TO_P_AU,
    (ny - 1.0 - YC_MANUAL) * PX2_TO_P_AU,
]

im1 = ax_exp.imshow(
    H_exp_pad,
    origin="lower",
    cmap="inferno",
    extent=exp_extent_pad_au,
    aspect="equal",
    vmin=0.0,
    vmax=exp_vmax,
)

# independent limits
ax_sim.set_xlim(-5, 5)
ax_sim.set_ylim(-5, 5)

ax_exp.set_xlim(-3, 3)
ax_exp.set_ylim(-3,3)

ax_sim.set_aspect("equal", adjustable="box")
ax_exp.set_aspect("equal", adjustable="box")
ax_sim.margins(0)
ax_exp.margins(0)

fig.colorbar(im0, cax=cax_sim)
fig.colorbar(im1, cax=cax_exp)
ax_sim.set_title("")
ax_exp.set_title("")

ax_sim.set_xlabel(r"$p_x$ [a.u.]", labelpad=2)
ax_sim.set_ylabel(r"$p_y$ [a.u.]", labelpad=2)

ax_exp.set_xlabel(r"$p_x$ [a.u.]", labelpad=2)
ax_exp.set_ylabel(r"", labelpad=2)

# -----------------------
# Middle row: mean single-shot max electron KE vs IR
# -----------------------
# -----------------------
# Simulation points to overlay in middle panel
# -----------------------
SIM_MID_POINTS = [
    {"h5": H5_NEUTRAL, "time_index": TIME_INDEX, "ir_mJ": 100, "label": r"$\mathrm{Sim. neutral}~\times (\frac{1}{3})$"},
]
if H5_IONIZED is not None:
    SIM_MID_POINTS.append({"h5": H5_IONIZED, "time_index": TIME_INDEX, "ir_mJ": 20, "label": r"$\mathrm{Sim. pre-ionized}$"})

SIM_Q_EDGE = 0.90   # 98% radial quantile -> max-like image observable

for T in temps:
    block = [s for s in summary if np.isfinite(s["ir"]) and s["temp"] == T and np.isfinite(s["r_mean"])]
    block = sorted(block, key=lambda d: d["ir"])
    if not block:
        continue

    x = np.array([b["ir"] for b in block])
    y = np.array([b["r_mean"] for b in block])
    yerr = np.array([b["r_std"] for b in block])

    if T == 12:
        Nlabel = 2e5
    elif T == 10:
        Nlabel = 2e6
    elif T == 9:
        Nlabel = 8e6
    else:
        Nlabel = np.nan

    y = PX2_TO_EV * y**2
    yerr = PX2_TO_EV * yerr**2

    ax_mid.plot(x, y, marker="o", lw=2, ms=8, label=sci_label(Nlabel))
    ax_mid.fill_between(x, y - yerr, y + yerr, alpha=0.15)
# -----------------------
# Add simulated VMI-based points to middle panel
# -----------------------
for pt in SIM_MID_POINTS:
    # max-like image observable from sim VMI
    E_sim_eV, p_sim_au = sim_vmi_edge_energy_eV(
        pt["h5"],
        pt["time_index"],
        q=SIM_Q_EDGE
    )

    if not np.isfinite(E_sim_eV):
        continue
    if pt["h5"]== H5_NEUTRAL:
        E_sim_eV/=3
        aux=(0, -18)
    if H5_IONIZED is not None and pt["h5"] == H5_IONIZED:
        aux=(0, 12)

    ax_mid.scatter(
        pt["ir_mJ"],
        E_sim_eV,
        s=110,
        marker="D",
        facecolor="white",
        edgecolor="black",
        linewidth=1.8,
        zorder=6,
    )
    ax_mid.annotate(
        pt["label"],
        (pt["ir_mJ"], E_sim_eV),
        xytext=aux,
        textcoords="offset points",
    )

    print(f"{pt['label']}: E_edge({SIM_Q_EDGE:.2f}) = {E_sim_eV:.3f} eV  at p = {p_sim_au:.3f} a.u.")
ax_mid.set_xlabel(rf"IR pulse energy ($\mu$J)")
ax_mid.set_ylabel(r"$\langle e^{-}\ \mathrm{KE}\rangle$ (eV)")
ax_mid.legend(title="Cluster size")

# -----------------------
# Bottom row: ion momentum
# summed simulation (t0 + t106) vs experiment
# -----------------------
steps = np.array([extract_step_from_filename(fn) for fn in run.files], dtype=int)
times_all_fs = np.array([run.time_fs_from_step(st) for st in steps], dtype=float)

# determine common momentum range from the two selected times
pmax_found = 0.0
sel_idx = []
for tt in SIM_SUM_TIMES_FS:
    i = int(np.argmin(np.abs(times_all_fs - tt)))
    if i not in sel_idx:
        sel_idx.append(i)

for i in sel_idx:
    fn = run.files[i]
    step = steps[i]
    px, py, pz, w = run._read_momentum_and_weight(fn, step, ION_SPECIES)
    if px is None:
        continue
    p_abs_au = np.sqrt(px*px + py*py + pz*pz) / p_au_SI
    if p_abs_au.size > 0:
        pmax_found = max(pmax_found, float(np.nanmax(p_abs_au)))

P_MAX_AU = 1.05 * pmax_found if pmax_found > 0 else 1.0
edges = np.linspace(0.0, P_MAX_AU, ION_NBINS + 1)
mid = 0.5 * (edges[1:] + edges[:-1])

# raw simulation histograms at the two times
t0_fs, h0 = get_hist_for_time(run, ION_SPECIES, SIM_SUM_TIMES_FS[0], edges, p_au_SI)
t1_fs, h1 = get_hist_for_time(run, ION_SPECIES, SIM_SUM_TIMES_FS[1], edges, p_au_SI)

# summed raw histogram, then normalize
h_sum = h0 + h1

if np.nanmax(h_sum) > 0:
    h_sum = h_sum / np.nanmax(h_sum)

# helper normalization for faint component curves
h0_plot = h0.copy()
h1_plot = h1.copy()
if np.nanmax(h0_plot) > 0:
    h0_plot /= np.nanmax(h0_plot)
if np.nanmax(h1_plot) > 0:
    h1_plot /= np.nanmax(h0)

# experimental curve
with h5py.File(EXP_ION_H5, "r") as h5:
    c_exp = h5[f"runs/run_{RUN_EXP_ION}/hist/counts_P"][:].astype(float)
    p_exp = h5[f"runs/run_{RUN_EXP_ION}/hist/centers_P"][:].astype(float)

if np.nanmax(c_exp) > 0:
    c_exp = c_exp / np.nanmax(c_exp)

# align summed simulation peak to experimental peak
p_peak_exp = p_exp[np.nanargmax(c_exp)]
p_peak_sum = mid[np.nanargmax(h_sum)]
shift_sum = p_peak_exp - p_peak_sum
mid_aligned = mid + shift_sum

# optional faint ingredient curves
if SHOW_COMPONENTS:
    ax_bot.plot(mid, h0_plot, lw=1.2, alpha=0.7, ls="--", label=f"Sim. t={t0_fs:.0f} fs")
    ax_bot.plot(mid, h1_plot, lw=1.2, alpha=0.7, ls=":", label=f"Sim. t={t1_fs:.0f} fs")

# summed + shifted sim curve
ax_bot.plot(
    mid_aligned, h_sum,
    lw=2.4,
    label=rf"$\left(\mathrm{{t}}_{{{t0_fs:.0f}\,\mathrm{{fs}}}} + \mathrm{{t}}_{{{t1_fs:.0f}\,\mathrm{{fs}}}}\right) (shifted)$"
)

# experiment
ax_bot.plot(p_exp, c_exp, "o-", ms=4, lw=1.8, label=EXP_ION_LABEL)

ax_bot.set_xlabel(r"$|p|$ (a.u.)")
ax_bot.set_ylabel("Normalized counts")
ax_bot.set_xlim(0, 300)
if LOGY:
    ax_bot.set_yscale("log")
ax_bot.grid(True, alpha=0.25)
ax_bot.legend(frameon=False)

plt.tight_layout()
plt.savefig("Fig5.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# =========================================================
# Simulated VMIs + circle radius / energy extraction
# - radius is obtained from the simulated VMI image itself
# - circle is drawn in momentum space (a.u.)
# - also reports equivalent energy (eV) and detector radius (px)
#   using the experimental calibration PX2_TO_EV
# =========================================================

import h5py
import numpy as np
import matplotlib.pyplot as plt
import scipy.constants as sc
from matplotlib.patches import Circle

# -----------------------
# Inputs
# -----------------------
SIM_VMI_CASES = [
    {"h5": H5_NEUTRAL, "time_index": TIME_INDEX, "title": "Neutral"},
    # {"h5": H5_IONIZED, "time_index": 18, "title": "Pre-ionized"},
]

# image-based "edge" definition:
# q = 0.98 means radius containing 98% of the VMI signal
SIM_Q_EDGE = 0.99

# optional display limits in a.u. (set to None for native range)
PLOT_XLIM = None   # e.g. (-5, 5)
PLOT_YLIM = None   # e.g. (-5, 5)

# experimental calibration used only to report an equivalent detector radius
PX2_TO_EV = 3.9914e-3

# color scaling
VMI_VMAX_PERCENTILE = 98

# -----------------------
# Constants
# -----------------------
EV_PER_HA = sc.physical_constants["Hartree energy in eV"][0]
P_AU_SI = sc.physical_constants["atomic unit of momentum"][0]

# -----------------------
# Helpers
# -----------------------
def load_sim_vmi(h5_path, time_index):
    with h5py.File(h5_path, "r") as h5:
        H = h5["spectra/He_e_H_pxy"][time_index, :, :].astype(float)
        p_mid_SI = h5["axes/He_e_p_mid_SI"][:]
    p_mid_au = p_mid_SI / P_AU_SI
    return H, p_mid_au

def radial_quantile_from_vmi(H, p_mid_au, q=0.98):
    """
    Radius p_q (a.u.) containing fraction q of the VMI signal.
    """
    PX, PY = np.meshgrid(p_mid_au, p_mid_au, indexing="xy")
    P = np.sqrt(PX**2 + PY**2).ravel()
    W = np.clip(H, 0.0, None).ravel()

    m = np.isfinite(P) & np.isfinite(W) & (W > 0)
    if not np.any(m):
        return np.nan

    P = P[m]
    W = W[m]

    order = np.argsort(P)
    P = P[order]
    W = W[order]

    cdf = np.cumsum(W)
    cdf = cdf / cdf[-1]

    return float(np.interp(q, cdf, P))

def momentum_au_to_energy_eV(p_au):
    return 0.5 * p_au**2 * EV_PER_HA

def energy_eV_to_radius_px(E_eV, px2_to_ev):
    if E_eV < 0:
        return np.nan
    return np.sqrt(E_eV / px2_to_ev)

# -----------------------
# Plot
# -----------------------
n = len(SIM_VMI_CASES)
fig, axes = plt.subplots(1, n, figsize=(5.3 * n, 5.2), squeeze=False)
axes = axes[0]

results = []

for ax, case in zip(axes, SIM_VMI_CASES):
    H, p_mid_au = load_sim_vmi(case["h5"], case["time_index"])

    dp = p_mid_au[1] - p_mid_au[0]
    extent_au = [
        p_mid_au[0]  - 0.5 * dp,
        p_mid_au[-1] + 0.5 * dp,
        p_mid_au[0]  - 0.5 * dp,
        p_mid_au[-1] + 0.5 * dp,
    ]

    p_q_au = radial_quantile_from_vmi(H, p_mid_au, q=SIM_Q_EDGE)
    E_q_eV = momentum_au_to_energy_eV(p_q_au)
    r_q_px = energy_eV_to_radius_px(E_q_eV, PX2_TO_EV)

    vmax = np.nanpercentile(H, VMI_VMAX_PERCENTILE) if np.any(np.isfinite(H)) else 1.0

    im = ax.imshow(
        H,
        origin="lower",
        cmap="inferno",
        extent=extent_au,
        aspect="equal",
        vmin=0.0,
        vmax=vmax,
    )

    # circle in momentum space
    circ = Circle((0.0, 0.0), p_q_au, fill=False, ec="cyan", lw=2.0, ls="--")
    ax.add_patch(circ)

    # center marker
    ax.plot(0.0, 0.0, marker="+", ms=10, mew=1.6, color="white")

    # limits
    if PLOT_XLIM is not None:
        ax.set_xlim(*PLOT_XLIM)
    if PLOT_YLIM is not None:
        ax.set_ylim(*PLOT_YLIM)

    ax.set_xlabel(r"$p_x$ [a.u.]")
    ax.set_ylabel(r"$p_y$ [a.u.]")
    ax.set_title(
        f"{case['title']}\n"
        rf"$p_{{{SIM_Q_EDGE:.2f}}}={p_q_au:.2f}$ a.u., "
        rf"$E={E_q_eV:.2f}$ eV"
    )

    txt = (
        rf"$q={SIM_Q_EDGE:.2f}$" "\n"
        rf"$p_q={p_q_au:.3f}$ a.u." "\n"
        rf"$E_q={E_q_eV:.3f}$ eV" "\n"
        rf"$r_q={r_q_px:.1f}$ px"
    )
    ax.text(
        0.03, 0.97, txt,
        transform=ax.transAxes,
        va="top", ha="left",
        fontsize=10,
        color="white",
        bbox=dict(boxstyle="round,pad=0.25", facecolor="black", alpha=0.55, edgecolor="none")
    )

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
    cbar.set_label("counts")

    results.append({
        "title": case["title"],
        "time_index": case["time_index"],
        "p_q_au": p_q_au,
        "E_q_eV": E_q_eV,
        "r_q_px": r_q_px,
    })

plt.tight_layout()
plt.show()

print("Extracted radii / energies from simulated VMI:")
for r in results:
    print(
        f"{r['title']:>12s} | time_index={r['time_index']:>3d} | "
        f"p_q={r['p_q_au']:.4f} a.u. | "
        f"E_q={r['E_q_eV']:.4f} eV | "
        f"equiv radius={r['r_q_px']:.2f} px"
    )


In [ ]:
# =========================================================
# CELL 1
# Total ion kinetic energy vs time from PIConGPU
#
# Computes:
#   E_tot(t)  = sum_i w_i * p_i^2 / (2m)
#   <E>(t)    = E_tot / sum_i w_i
#
# Optional:
#   set Z_FILTER_MIN = 1 to include only charged ions.
#   set Z_FILTER_MIN = None to include all He_i particles.
# =========================================================

import numpy as np
import matplotlib.pyplot as plt
import scipy.constants as sc
import adios2

from nanoplasma_analysis import NanoPlasmaRun, extract_step_from_filename

# ---------------------------------------------------------
# USER INPUTS
# ---------------------------------------------------------
run = NanoPlasmaRun(
    path=str(SIM_OUTPUT),
    laser_peak_at_target=89603,
)

ION_SPECIES = "He_i"
ZMAX = 2

# None = all He_i particles
# 1    = only charged ions, Z >= 1
Z_FILTER_MIN = None

SAVE_FIG = False
OUTNAME = "total_ion_kinetic_energy_vs_time.png"

# ---------------------------------------------------------
# Helper: read bound-electron charge state if needed
# ---------------------------------------------------------
def read_charge_state_Z(filename, step, species="He_i", zmax=2):
    with adios2.Stream(filename, "r") as f:
        for _ in f.steps():
            base = f"/data/{step}/particles/{species}"
            be = f.read(f"{base}/boundElectrons")
            Z = zmax - np.asarray(be, dtype=float)
            return Z
    return None

# ---------------------------------------------------------
# Loop over all outputs
# ---------------------------------------------------------
rows = []

m_ion = run._mass_kg(ION_SPECIES)

for i, fn in enumerate(run.files):
    step = extract_step_from_filename(fn)
    t_fs = run.time_fs_from_step(step)

    px, py, pz, w = run._read_momentum_and_weight(fn, step, ION_SPECIES)
    if px is None or w is None:
        continue

    px = np.asarray(px, dtype=float)
    py = np.asarray(py, dtype=float)
    pz = np.asarray(pz, dtype=float)
    w  = np.asarray(w, dtype=float)

    E_eV = (px**2 + py**2 + pz**2) / (2.0 * m_ion) / sc.e

    if Z_FILTER_MIN is not None:
        try:
            Z = read_charge_state_Z(fn, step, ION_SPECIES, ZMAX)
            mask = Z >= Z_FILTER_MIN
        except Exception as err:
            print(f"Could not read Z at step {step}: {err}")
            mask = np.ones_like(w, dtype=bool)
    else:
        Z = None
        mask = np.ones_like(w, dtype=bool)

    if np.sum(mask) == 0:
        rows.append({
            "file_index": i,
            "step": step,
            "time_fs": t_fs,
            "total_E_eV": 0.0,
            "mean_E_eV": np.nan,
            "total_weight": 0.0,
        })
        continue

    w_sel = w[mask]
    E_sel = E_eV[mask]

    total_weight = np.sum(w_sel)
    total_E_eV = np.sum(w_sel * E_sel)
    mean_E_eV = total_E_eV / (total_weight + 1e-300)

    rows.append({
        "file_index": i,
        "step": step,
        "time_fs": t_fs,
        "total_E_eV": total_E_eV,
        "mean_E_eV": mean_E_eV,
        "total_weight": total_weight,
    })

if len(rows) == 0:
    raise RuntimeError("No ion kinetic-energy data found.")

# Sort by time
rows = sorted(rows, key=lambda d: d["time_fs"])

t_fs = np.array([d["time_fs"] for d in rows])
E_tot_eV = np.array([d["total_E_eV"] for d in rows])
E_mean_eV = np.array([d["mean_E_eV"] for d in rows])
N_weight = np.array([d["total_weight"] for d in rows])

# ---------------------------------------------------------
# Print useful numbers
# ---------------------------------------------------------
print(f"Number of valid outputs: {len(rows)}")
print(f"Time range: {t_fs[0]:.2f} fs to {t_fs[-1]:.2f} fs")
print(f"Z filter: {Z_FILTER_MIN}")
print(f"Final total ion KE: {E_tot_eV[-1]:.6e} eV")
print(f"Final mean ion KE:  {E_mean_eV[-1]:.6g} eV")

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------
fig, ax1 = plt.subplots(figsize=(7.0, 4.2))

ax1.plot(t_fs, E_tot_eV, "o-", ms=4, lw=1.8, label="Total ion KE")
ax1.set_xlabel("time - laser peak (fs)")
ax1.set_ylabel("Total ion kinetic energy (eV)")
ax1.grid(True, alpha=0.25)

ax2 = ax1.twinx()
ax2.plot(t_fs, E_mean_eV, "s--", ms=3.5, lw=1.5, alpha=0.75, label="Mean ion KE")
ax2.set_ylabel("Mean ion kinetic energy (eV)")

title_filter = "all He_i particles" if Z_FILTER_MIN is None else rf"charged ions, $Z\geq {Z_FILTER_MIN}$"
ax1.set_title(f"Ion kinetic energy vs time ({title_filter})")

# combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")

plt.tight_layout()

if SAVE_FIG:
    fig.savefig(OUTNAME, dpi=300, bbox_inches="tight")

plt.show()

# Save result for later
ion_total_ke_result = {
    "rows": rows,
    "time_fs": t_fs,
    "total_E_eV": E_tot_eV,
    "mean_E_eV": E_mean_eV,
    "total_weight": N_weight,
    "Z_FILTER_MIN": Z_FILTER_MIN,
}

print("Saved dictionary: ion_total_ke_result")


In [ ]:
# =========================================================
# CELL 1b
# Fit total ion kinetic energy vs time with saturation models
#
# Requires Cell 1 output:
#   ion_total_ke_result
#
# Models:
#   1) logistic sigmoid
#   2) Gompertz sigmoid
#   3) Weibull / stretched-exponential ramp
#
# Goal:
#   estimate final/converged total ion kinetic energy E_inf
# =========================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# ---------------------------------------------------------
# USER INPUTS
# ---------------------------------------------------------
# Choose what to fit:
#   "total_E_eV" = total ion kinetic energy
#   "mean_E_eV"  = mean kinetic energy per ion
FIT_QUANTITY = "total_E_eV"

# Time range used for fit
FIT_TMIN_FS = -20.0
FIT_TMAX_FS = None     # None = use all available data

# How far beyond the simulation to plot the extrapolation
EXTRAPOLATE_TO_FS = 300.0

# Bound on final energy.
# If the simulation does not reach a plateau, this prevents absurd E_inf.
# Increase if the fit hits the upper bound.
E_INF_MAX_FACTOR = 20.0

SAVE_FIG = False
OUTNAME = "total_ion_kinetic_energy_sigmoid_fit.png"

# ---------------------------------------------------------
# Read data from Cell 1
# ---------------------------------------------------------
t_all = np.asarray(ion_total_ke_result["time_fs"], dtype=float)
y_all = np.asarray(ion_total_ke_result[FIT_QUANTITY], dtype=float)

m = np.isfinite(t_all) & np.isfinite(y_all) & (y_all >= 0)

if FIT_TMIN_FS is not None:
    m &= t_all >= FIT_TMIN_FS

if FIT_TMAX_FS is not None:
    m &= t_all <= FIT_TMAX_FS

t = t_all[m]
y = y_all[m]

order = np.argsort(t)
t = t[order]
y = y[order]

if len(t) < 6:
    raise RuntimeError("Not enough valid points to fit.")

ymax_data = np.nanmax(y)
ymin_data = np.nanmin(y)

print(f"Fitting quantity: {FIT_QUANTITY}")
print(f"Number of fit points: {len(t)}")
print(f"Data time range: {t[0]:.2f} fs to {t[-1]:.2f} fs")
print(f"Latest value: {y[-1]:.6e}")

# ---------------------------------------------------------
# Models
# ---------------------------------------------------------
def logistic_model(t, y0, yinf, t50, tau):
    """
    Symmetric sigmoid.
    t50 is the inflection point and also the 50% point.
    """
    tau = np.maximum(tau, 1e-12)
    return y0 + (yinf - y0) / (1.0 + np.exp(-(t - t50) / tau))

def gompertz_model(t, y0, yinf, tinfl, tau):
    """
    Asymmetric sigmoid.
    tinfl is the inflection time.
    """
    tau = np.maximum(tau, 1e-12)
    return y0 + (yinf - y0) * np.exp(-np.exp(-(t - tinfl) / tau))

def weibull_ramp_model(t, y0, yinf, t0, tau, beta):
    """
    Smooth ignition / stretched-exponential ramp.

    y = y0                           for t <= t0
    y = y0 + (yinf-y0)(1-exp(-z^beta)) for t > t0

    beta > 1 gives a slow start followed by an inflection.
    """
    tau = np.maximum(tau, 1e-12)
    beta = np.maximum(beta, 1e-12)

    z = np.maximum((t - t0) / tau, 0.0)
    return y0 + (yinf - y0) * (1.0 - np.exp(-(z ** beta)))

# ---------------------------------------------------------
# Fit helper
# ---------------------------------------------------------
def fit_model(name, func, p0, bounds):
    try:
        popt, pcov = curve_fit(
            func,
            t,
            y,
            p0=p0,
            bounds=bounds,
            maxfev=200000,
        )

        yfit = func(t, *popt)
        resid = y - yfit

        rss = np.sum(resid**2)
        n = len(y)
        k = len(popt)

        # AIC for relative comparison
        aic = n * np.log(rss / n + 1e-300) + 2 * k

        return {
            "name": name,
            "func": func,
            "popt": popt,
            "pcov": pcov,
            "success": True,
            "rss": rss,
            "aic": aic,
            "resid": resid,
            "yfit": yfit,
        }

    except Exception as err:
        print(f"{name} fit failed: {err}")
        return {
            "name": name,
            "func": func,
            "popt": None,
            "pcov": None,
            "success": False,
            "rss": np.inf,
            "aic": np.inf,
            "resid": None,
            "yfit": None,
        }

# ---------------------------------------------------------
# Initial guesses and bounds
# ---------------------------------------------------------
y0_guess = max(0.0, ymin_data)
yinf_guess = max(1.2 * y[-1], ymax_data)
yinf_upper = E_INF_MAX_FACTOR * max(y[-1], ymax_data, 1e-300)

# crude inflection guess: point of largest derivative
dy_dt = np.gradient(y, t)
i_infl_guess = int(np.nanargmax(dy_dt))
t_infl_guess = t[i_infl_guess]

tau_guess = max((t[-1] - t[0]) / 6.0, 5.0)

fits = []

# Logistic
fits.append(
    fit_model(
        "logistic",
        logistic_model,
        p0=[y0_guess, yinf_guess, t_infl_guess, tau_guess],
        bounds=(
            [0.0,       ymax_data,      t[0] - 100.0,  0.1],
            [ymax_data, yinf_upper,     t[-1] + 300.0, 500.0],
        ),
    )
)

# Gompertz
fits.append(
    fit_model(
        "gompertz",
        gompertz_model,
        p0=[y0_guess, yinf_guess, t_infl_guess, tau_guess],
        bounds=(
            [0.0,       ymax_data,      t[0] - 100.0,  0.1],
            [ymax_data, yinf_upper,     t[-1] + 300.0, 500.0],
        ),
    )
)

# Weibull / stretched exponential
fits.append(
    fit_model(
        "weibull_ramp",
        weibull_ramp_model,
        p0=[y0_guess, yinf_guess, t[0], tau_guess, 2.0],
        bounds=(
            [0.0,       ymax_data,      t[0] - 100.0,  0.1, 0.3],
            [ymax_data, yinf_upper,     t[-1] + 100.0, 800.0, 8.0],
        ),
    )
)

# Keep successful fits
fits_success = [f for f in fits if f["success"]]

if len(fits_success) == 0:
    raise RuntimeError("All saturation fits failed.")

# Best by AIC
best = min(fits_success, key=lambda d: d["aic"])

# ---------------------------------------------------------
# Derived quantities
# ---------------------------------------------------------
def time_to_fraction_logistic(y0, yinf, t50, tau, frac):
    return t50 + tau * np.log(frac / (1.0 - frac))

def time_to_fraction_gompertz(y0, yinf, tinfl, tau, frac):
    return tinfl - tau * np.log(-np.log(frac))

def time_to_fraction_weibull(y0, yinf, t0, tau, beta, frac):
    return t0 + tau * (-np.log(1.0 - frac)) ** (1.0 / beta)

def inflection_weibull(y0, yinf, t0, tau, beta):
    if beta <= 1:
        return np.nan
    return t0 + tau * ((beta - 1.0) / beta) ** (1.0 / beta)

def summarize_fit(f):
    name = f["name"]
    p = f["popt"]

    if name == "logistic":
        y0, yinf, t50, tau = p
        tinfl = t50
        t90 = time_to_fraction_logistic(y0, yinf, t50, tau, 0.90)
        t95 = time_to_fraction_logistic(y0, yinf, t50, tau, 0.95)
        param_text = f"y0={y0:.4e}, E_inf={yinf:.4e}, t50={t50:.2f} fs, tau={tau:.2f} fs"

    elif name == "gompertz":
        y0, yinf, tinfl, tau = p
        t90 = time_to_fraction_gompertz(y0, yinf, tinfl, tau, 0.90)
        t95 = time_to_fraction_gompertz(y0, yinf, tinfl, tau, 0.95)
        param_text = f"y0={y0:.4e}, E_inf={yinf:.4e}, t_infl={tinfl:.2f} fs, tau={tau:.2f} fs"

    elif name == "weibull_ramp":
        y0, yinf, t0, tau, beta = p
        tinfl = inflection_weibull(y0, yinf, t0, tau, beta)
        t90 = time_to_fraction_weibull(y0, yinf, t0, tau, beta, 0.90)
        t95 = time_to_fraction_weibull(y0, yinf, t0, tau, beta, 0.95)
        param_text = (
            f"y0={y0:.4e}, E_inf={yinf:.4e}, "
            f"t0={t0:.2f} fs, tau={tau:.2f} fs, beta={beta:.2f}"
        )

    else:
        tinfl, t90, t95, param_text = np.nan, np.nan, np.nan, ""

    return {
        "E_inf": yinf,
        "t_infl": tinfl,
        "t90": t90,
        "t95": t95,
        "param_text": param_text,
    }

print("\nFit summary:")
for f in fits_success:
    s = summarize_fit(f)
    hit_upper = np.isclose(s["E_inf"], yinf_upper, rtol=1e-3, atol=0)
    warn = "  <-- WARNING: E_inf near upper bound" if hit_upper else ""
    print(f"\n{f['name']}")
    print(f"  {s['param_text']}")
    print(f"  AIC = {f['aic']:.3g}, RSS = {f['rss']:.3e}")
    print(f"  inflection ≈ {s['t_infl']:.2f} fs")
    print(f"  t90 ≈ {s['t90']:.2f} fs, t95 ≈ {s['t95']:.2f} fs{warn}")

print(f"\nBest model by AIC: {best['name']}")

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------
t_plot_max = max(EXTRAPOLATE_TO_FS, t[-1])
t_plot = np.linspace(t[0], t_plot_max, 1200)

fig, axes = plt.subplots(2, 1, figsize=(7.2, 7.0), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1]})

# Top: data + fits
ax = axes[0]

ax.plot(t, y, "ko", ms=4, label="simulation data")

for f in fits_success:
    y_plot = f["func"](t_plot, *f["popt"])
    s = summarize_fit(f)

    lw = 2.8 if f["name"] == best["name"] else 1.7
    alpha = 1.0 if f["name"] == best["name"] else 0.65

    ax.plot(
        t_plot,
        y_plot,
        lw=lw,
        alpha=alpha,
        label=rf"{f['name']}: $E_\infty={s['E_inf']:.3e}$"
    )

    ax.axhline(s["E_inf"], ls="--", lw=1.0, alpha=0.35)

ax.axvline(t_infl_guess, color="gray", ls=":", lw=1.2,
           label=rf"max slope guess $\sim {t_infl_guess:.0f}$ fs")

ax.set_ylabel("Total ion KE (eV)" if FIT_QUANTITY == "total_E_eV" else "Mean ion KE (eV)")
ax.set_title("Saturation fit of ion kinetic energy")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

# Bottom: residuals
ax = axes[1]

for f in fits_success:
    ax.plot(t, f["resid"], "o-", ms=3, lw=1.2, label=f["name"])

ax.axhline(0, color="k", lw=1.0)
ax.set_xlabel("time - laser peak (fs)")
ax.set_ylabel("residual")
ax.grid(True, alpha=0.25)

plt.tight_layout()

if SAVE_FIG:
    fig.savefig(OUTNAME, dpi=300, bbox_inches="tight")

plt.show()

# ---------------------------------------------------------
# Save result
# ---------------------------------------------------------
ion_ke_saturation_fit_result = {
    "FIT_QUANTITY": FIT_QUANTITY,
    "t_fit_fs": t,
    "y_fit": y,
    "fits": fits,
    "fits_success": fits_success,
    "best": best,
    "best_summary": summarize_fit(best),
    "t_plot_fs": t_plot,
}

print("\nSaved dictionary: ion_ke_saturation_fit_result")


In [ ]:
# =========================================================
# Saalmann + Bychenkov-Kovalev final ion KE distributions
# using:
#   - initial droplet density from the simulation
#   - known final mean ion kinetic energy
#
# This does NOT fit anything.
# It predicts final dN/dE shapes from Coulomb-explosion models.
#
# Saalmann:
#   homogeneous sphere, dP/dE ∝ sqrt(E/Emax)
#
# Bychenkov-Kovalev:
#   arbitrary spherical density n(r)
#   Qenc(r) = ∫ n(r') r'^2 dr'
#   E(r) ∝ Qenc(r)/r
#
# All curves are scaled to the same final mean ion energy.
# =========================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

from nanoplasma_analysis import NanoPlasmaRun, extract_step_from_filename

# ---------------------------------------------------------
# USER INPUTS
# ---------------------------------------------------------
run = NanoPlasmaRun(
    path=str(SIM_OUTPUT),
    laser_peak_at_target=89603,
)

ION_SPECIES = "He_i"

# Use your measured/extrapolated final mean ion KE here.
MEAN_FINAL_ENERGY_EV = 51.0

# Optional consistency check with total energy
TOTAL_FINAL_ENERGY_EV = 3.0e7

# Which simulation output to use as the initial density.
# Use None for first valid ion frame.
INITIAL_FILE_INDEX = None

# Radius used for the model.
# "r95", "r99", or "rmax"
R0_MODE = "r95"

# radial histogram for extracting initial density
N_R_BINS = 600

# energy histogram for final spectra
E_MAX_EV = 100.0
N_E_BINS = 500

# smoothing only for visual finite-bin noise
SMOOTH_SIGMA_BINS = 1.0

SAVE_FIG = False
OUTNAME = "Saalmann_Bychenkov_initial_density_final_KE.png"

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def trapz_compat(y, x):
    if hasattr(np, "trapezoid"):
        return np.trapezoid(y, x)
    return np.trapz(y, x)

def normalize_max(y):
    y = np.asarray(y, dtype=float)
    m = np.nanmax(y)
    return y / m if np.isfinite(m) and m > 0 else y

def normalize_area(y, x):
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    area = trapz_compat(np.clip(y, 0, None), x)
    return y / area if np.isfinite(area) and area > 0 else y

def weighted_percentile(x, w, q):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)

    m = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x = x[m]
    w = w[m]

    if len(x) == 0:
        return np.nan

    order = np.argsort(x)
    x = x[order]
    w = w[order]

    cw = np.cumsum(w)
    return np.interp(q / 100.0 * cw[-1], cw, x)

def get_first_valid_position_frame(run, species):
    for i, fn in enumerate(run.files):
        step = extract_step_from_filename(fn)
        x, y, z, w = run._read_positions_and_weight(fn, step, species)
        if x is not None and w is not None and np.sum(w) > 0:
            return i
    raise RuntimeError(f"No valid position data found for species={species}")

def read_initial_radial_distribution(run, species, file_index=None, r0_mode="r99", n_r_bins=600):
    if file_index is None:
        file_index = get_first_valid_position_frame(run, species)

    fn = run.files[file_index]
    step = extract_step_from_filename(fn)
    t_fs = run.time_fs_from_step(step)

    x, y, z, w = run._read_positions_and_weight(fn, step, species)
    if x is None:
        raise RuntimeError(f"No position data for {species} at file index {file_index}")

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    z = np.asarray(z, dtype=float)
    w = np.asarray(w, dtype=float)

    wsum = np.sum(w)

    # center of droplet
    x0 = np.sum(w * x) / (wsum + 1e-300)
    y0 = np.sum(w * y) / (wsum + 1e-300)
    z0 = np.sum(w * z) / (wsum + 1e-300)

    r_nm = np.sqrt((x - x0)**2 + (y - y0)**2 + (z - z0)**2) * 1e9

    if r0_mode == "r95":
        R0_nm = weighted_percentile(r_nm, w, 95)
    elif r0_mode == "r99":
        R0_nm = weighted_percentile(r_nm, w, 99)
    elif r0_mode == "rmax":
        R0_nm = np.nanmax(r_nm)
    else:
        raise ValueError("R0_MODE must be 'r95', 'r99', or 'rmax'.")

    r_edges = np.linspace(0.0, R0_nm, n_r_bins + 1)
    r_mid = 0.5 * (r_edges[1:] + r_edges[:-1])
    dr = np.diff(r_edges)

    # Only keep particles inside model radius
    hist_shell, _ = np.histogram(r_nm, bins=r_edges, weights=w)

    # radial number density, up to units
    shell_volume = 4.0 * np.pi * np.maximum(r_mid, 1e-12)**2 * dr
    n_r = hist_shell / np.maximum(shell_volume, 1e-300)

    return {
        "file_index": file_index,
        "step": step,
        "time_fs": t_fs,
        "R0_nm": R0_nm,
        "r_mid_nm": r_mid,
        "r_edges_nm": r_edges,
        "shell_weights": hist_shell,
        "density_profile": n_r,
        "total_weight_inside_R0": np.sum(hist_shell),
        "total_weight_all": wsum,
    }

def saalmann_single_distribution(E_mid, mean_energy_eV):
    """
    Homogeneous charged sphere:
      dP/dε = 3/2 sqrt(ε), 0 < ε < 1
      <E> = 3/5 Emax
    """
    Emax = (5.0 / 3.0) * mean_energy_eV
    eps = np.clip(E_mid / Emax, 0.0, None)

    y = 1.5 * np.sqrt(eps) * (eps <= 1.0)
    y = normalize_area(y, E_mid)

    return y, Emax

def bychenkov_from_shell_weights(r_mid_nm, shell_weights, E_edges, mean_energy_target_eV):
    """
    Build BK spectrum from discrete radial shell weights.

    shell_weights ~ dN in each radial shell.
    Qenc(r) is cumulative shell charge/number.
    E_shape(r) ∝ Qenc(r)/r.
    Then scale E_shape so that <E> = mean_energy_target_eV.
    """
    r_mid_nm = np.asarray(r_mid_nm, dtype=float)
    dN_shell = np.asarray(shell_weights, dtype=float)

    m = np.isfinite(r_mid_nm) & np.isfinite(dN_shell) & (r_mid_nm > 0) & (dN_shell >= 0)
    r = r_mid_nm[m]
    dN = dN_shell[m]

    if np.sum(dN) <= 0:
        raise RuntimeError("Empty radial distribution.")

    # Normalize total number/charge to 1 for shape calculation
    dN = dN / np.sum(dN)

    Qenc = np.cumsum(dN)

    # Coulomb explosion energy shape
    E_shape = Qenc / np.maximum(r, 1e-12)

    # Smooth start: force center close to zero if needed
    E_shape = np.clip(E_shape, 0, None)

    mean_shape = np.sum(dN * E_shape)
    scale = mean_energy_target_eV / max(mean_shape, 1e-300)

    E_shell = scale * E_shape

    hist, _ = np.histogram(E_shell, bins=E_edges, weights=dN)
    dNdE = hist / np.diff(E_edges)

    if SMOOTH_SIGMA_BINS is not None and SMOOTH_SIGMA_BINS > 0:
        dNdE = gaussian_filter1d(dNdE.astype(float), SMOOTH_SIGMA_BINS, mode="nearest")

    E_mid = 0.5 * (E_edges[1:] + E_edges[:-1])
    dNdE = normalize_area(dNdE, E_mid)

    mean_check = trapz_compat(E_mid * dNdE, E_mid)

    return {
        "E_mid": E_mid,
        "dNdE": dNdE,
        "E_shell": E_shell,
        "r_shell_nm": r,
        "dN_shell_norm": dN,
        "Qenc_norm": Qenc,
        "mean_check_eV": mean_check,
        "Emax_eV": np.nanmax(E_shell),
        "scale": scale,
    }

def analytic_profile_shells(kind, R0_nm, n_bins, **params):
    """
    Generate idealized profiles for comparison, same fixed R0.
    """
    r_edges = np.linspace(0.0, R0_nm, n_bins + 1)
    r_mid = 0.5 * (r_edges[1:] + r_edges[:-1])
    dr = np.diff(r_edges)

    x = r_mid / R0_nm

    if kind == "uniform":
        n = np.ones_like(x)

    elif kind == "gaussian":
        sigma = params.get("sigma", 0.45)
        n = np.exp(-0.5 * (x / sigma)**2)

    elif kind == "soft_edge":
        beta = params.get("beta", 2.0)
        n = np.maximum(1.0 - x**2, 0.0)**beta

    elif kind == "shell":
        x0 = params.get("x0", 0.75)
        sigma = params.get("sigma", 0.15)
        n = np.exp(-0.5 * ((x - x0) / sigma)**2)

    else:
        raise ValueError(kind)

    shell_weights = n * 4.0 * np.pi * r_mid**2 * dr

    return r_mid, shell_weights, n

# =========================================================
# 1) Use final energy scale from simulation
# =========================================================
N_eff_from_energy = TOTAL_FINAL_ENERGY_EV / MEAN_FINAL_ENERGY_EV

print("Energy-scale input:")
print(f"  Total final ion KE = {TOTAL_FINAL_ENERGY_EV:.4e} eV")
print(f"  Mean final ion KE  = {MEAN_FINAL_ENERGY_EV:.4g} eV")
print(f"  Effective ion number from total/mean = {N_eff_from_energy:.4e}")

# =========================================================
# 2) Read initial density from simulation
# =========================================================
init = read_initial_radial_distribution(
    run,
    ION_SPECIES,
    file_index=INITIAL_FILE_INDEX,
    r0_mode=R0_MODE,
    n_r_bins=N_R_BINS,
)

print("\nInitial density used:")
print(f"  file index = {init['file_index']}")
print(f"  step       = {init['step']}")
print(f"  time       = {init['time_fs']:.2f} fs")
print(f"  R0({R0_MODE}) = {init['R0_nm']:.3f} nm")
print(f"  weight inside R0 = {init['total_weight_inside_R0']:.4e}")
print(f"  total weight all  = {init['total_weight_all']:.4e}")

# =========================================================
# 3) Build Saalmann and Bychenkov spectra
# =========================================================
E_edges = np.linspace(0.0, E_MAX_EV, N_E_BINS + 1)
E_mid = 0.5 * (E_edges[1:] + E_edges[:-1])

# Saalmann homogeneous sphere
y_saalmann, Emax_saalmann = saalmann_single_distribution(
    E_mid,
    mean_energy_eV=MEAN_FINAL_ENERGY_EV,
)

# BK from actual initial simulation density
bk_sim = bychenkov_from_shell_weights(
    init["r_mid_nm"],
    init["shell_weights"],
    E_edges,
    mean_energy_target_eV=MEAN_FINAL_ENERGY_EV,
)

# BK analytic comparison profiles, same R0 and same mean energy
profile_results = []

profile_specs = [
    ("uniform",   "BK uniform", {}),
    ("gaussian", "BK center-peaked Gaussian", {"sigma": 0.45}),
    ("soft_edge", "BK soft edge", {"beta": 2.0}),
    ("shell",    "BK shell-like", {"x0": 0.75, "sigma": 0.15}),
]

for kind, label, pars in profile_specs:
    r_mid_prof, shell_w_prof, n_prof = analytic_profile_shells(
        kind,
        R0_nm=init["R0_nm"],
        n_bins=N_R_BINS,
        **pars,
    )

    bk_prof = bychenkov_from_shell_weights(
        r_mid_prof,
        shell_w_prof,
        E_edges,
        mean_energy_target_eV=MEAN_FINAL_ENERGY_EV,
    )

    bk_prof["label"] = label
    bk_prof["density_profile"] = n_prof
    bk_prof["r_mid_nm"] = r_mid_prof
    profile_results.append(bk_prof)

print("\nPredicted energy scales:")
print(f"  Saalmann Emax = {Emax_saalmann:.3f} eV")
print(f"  BK sim-density mean check = {bk_sim['mean_check_eV']:.3f} eV")
print(f"  BK sim-density Emax       = {bk_sim['Emax_eV']:.3f} eV")

# =========================================================
# 4) Plot
# =========================================================
fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.3))

# ---------------------------------------------------------
# Panel A: density profiles
# ---------------------------------------------------------
ax = axes[0]

ax.plot(
    init["r_mid_nm"],
    normalize_max(init["density_profile"]),
    lw=2.6,
    label="simulation initial density",
)

for prof in profile_results:
    ax.plot(
        prof["r_mid_nm"],
        normalize_max(prof["density_profile"]),
        lw=1.7,
        alpha=0.75,
        label=prof["label"].replace("BK ", ""),
    )

ax.set_xlabel("r (nm)")
ax.set_ylabel("normalized density")
ax.set_title("Initial radial density profiles")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=7)

# ---------------------------------------------------------
# Panel B: Coulomb energy mapping E(r)
# ---------------------------------------------------------
ax = axes[1]

ax.plot(
    bk_sim["r_shell_nm"],
    bk_sim["E_shell"],
    lw=2.6,
    label="simulation initial density",
)

for prof in profile_results:
    ax.plot(
        prof["r_shell_nm"],
        prof["E_shell"],
        lw=1.7,
        alpha=0.75,
        label=prof["label"],
    )

ax.set_xlabel("r (nm)")
ax.set_ylabel("final shell energy E(r) (eV)")
ax.set_title(r"BK mapping: $E(r)\propto Q_{\rm enc}(r)/r$")
ax.grid(True, alpha=0.25)

# ---------------------------------------------------------
# Panel C: final KE distributions
# ---------------------------------------------------------
ax = axes[2]

ax.plot(
    E_mid,
    normalize_max(y_saalmann),
    "k--",
    lw=2.0,
    label=rf"Saalmann uniform, $E_{{max}}={Emax_saalmann:.1f}$ eV",
)

ax.plot(
    bk_sim["E_mid"],
    normalize_max(bk_sim["dNdE"]),
    lw=2.8,
    label="BK from simulation initial density",
)

for prof in profile_results:
    ax.plot(
        prof["E_mid"],
        normalize_max(prof["dNdE"]),
        lw=1.7,
        alpha=0.75,
        label=prof["label"],
    )

ax.set_xlabel("Ion kinetic energy (eV)")
ax.set_ylabel(r"normalized $dN/dE$")
ax.set_xlim(0, E_MAX_EV)
ax.set_ylim(0, 1.08)
ax.set_title(r"Predicted final ion KE distributions")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=7)

plt.tight_layout()

if SAVE_FIG:
    fig.savefig(OUTNAME, dpi=300, bbox_inches="tight")

plt.show()

# =========================================================
# Save result for later
# =========================================================
analytical_final_ion_ke_result = {
    "MEAN_FINAL_ENERGY_EV": MEAN_FINAL_ENERGY_EV,
    "TOTAL_FINAL_ENERGY_EV": TOTAL_FINAL_ENERGY_EV,
    "N_eff_from_energy": N_eff_from_energy,
    "init_density": init,
    "E_mid": E_mid,
    "saalmann": {
        "dNdE": y_saalmann,
        "Emax_eV": Emax_saalmann,
    },
    "bk_sim_density": bk_sim,
    "bk_profiles": profile_results,
}

print("\nSaved dictionary: analytical_final_ion_ke_result")


In [ ]:
# =========================================================
# QUICK CELL
# Overlay experimental ion data on the analytical
# energy distributions from analytical_final_ion_ke_result
#
# Requires previous cell output:
#   analytical_final_ion_ke_result
# =========================================================

import h5py
import numpy as np
import matplotlib.pyplot as plt
import scipy.constants as sc

# ---------------------------------------------------------
# USER INPUTS
# ---------------------------------------------------------
EXP_ION_H5 = EXP_ION_H5
RUN_EXP_ION = 236

E_XMAX_EV = 100.0

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def trapz_compat(y, x):
    if hasattr(np, "trapezoid"):
        return np.trapezoid(y, x)
    return np.trapz(y, x)

def normalize_max(y):
    y = np.asarray(y, dtype=float)
    m = np.nanmax(y)
    return y / m if np.isfinite(m) and m > 0 else y

def centers_to_edges(x):
    x = np.asarray(x, dtype=float)
    dx = np.diff(x)
    edges = np.empty(len(x) + 1, dtype=float)
    edges[1:-1] = 0.5 * (x[:-1] + x[1:])
    edges[0] = x[0] - 0.5 * dx[0]
    edges[-1] = x[-1] + 0.5 * dx[-1]
    return edges

P_AU_SI = sc.physical_constants["atomic unit of momentum"][0]
M_ION_KG = 4.0 * sc.atomic_mass   # helium ion mass

def p_au_to_E_eV(p_au):
    p_si = np.asarray(p_au, dtype=float) * P_AU_SI
    return p_si**2 / (2.0 * M_ION_KG) / sc.e

def try_read_direct_energy_hist(hist):
    """
    Try to read a direct energy histogram from the experimental H5.
    """
    candidates = [
        ("edges_E_eV",  "centers_E_eV",  "counts_E"),
        ("edges_E",     "centers_E",     "counts_E"),
        ("edges_KE_eV", "centers_KE_eV", "counts_KE"),
        ("edges_KE",    "centers_KE",    "counts_KE"),
        ("E_edges",     "E_centers",     "counts_E"),
        ("energy_edges","energy_centers","counts_E"),
    ]

    for edge_key, center_key, count_key in candidates:
        if center_key in hist and count_key in hist:
            E_centers = hist[center_key][:].astype(float)
            counts_E = hist[count_key][:].astype(float)

            if edge_key in hist:
                E_edges = hist[edge_key][:].astype(float)
            else:
                E_edges = centers_to_edges(E_centers)

            dE = np.diff(E_edges)
            dE[dE <= 0] = np.nan
            dNdE = counts_E / dE

            return {
                "E_centers": E_centers,
                "E_edges": E_edges,
                "dNdE": dNdE,
                "source": f"direct energy histogram: {center_key}, {count_key}",
            }

    return None

# ---------------------------------------------------------
# Read experimental distribution
# ---------------------------------------------------------
with h5py.File(EXP_ION_H5, "r") as h5:
    hist = h5[f"runs/run_{RUN_EXP_ION}/hist"]

    print("Available experimental hist keys:")
    for k in hist.keys():
        print(" ", k)

    exp_direct = try_read_direct_energy_hist(hist)

    if exp_direct is None:
        print("\nNo direct energy histogram found. Converting from momentum histogram.")

        if "centers_P" not in hist or "counts_P" not in hist:
            raise KeyError("Need either direct energy histogram or momentum histogram (centers_P, counts_P).")

        p_centers = hist["centers_P"][:].astype(float)
        counts_P = hist["counts_P"][:].astype(float)

        if "edges_P" in hist:
            p_edges = hist["edges_P"][:].astype(float)
        else:
            p_edges = centers_to_edges(p_centers)

        p_edges = np.clip(p_edges, 0.0, None)

        E_edges = p_au_to_E_eV(p_edges)
        E_centers = 0.5 * (E_edges[1:] + E_edges[:-1])

        dE = np.diff(E_edges)
        dE[dE <= 0] = np.nan

        dNdE_exp = counts_P / dE
        exp_source = "converted from momentum histogram using Jacobian"
    else:
        E_centers = exp_direct["E_centers"]
        dNdE_exp = exp_direct["dNdE"]
        exp_source = exp_direct["source"]

print(f"\nExperimental source used: {exp_source}")

# ---------------------------------------------------------
# Get analytical curves from previous cell
# ---------------------------------------------------------
E_mid = analytical_final_ion_ke_result["E_mid"]

y_saalmann = analytical_final_ion_ke_result["saalmann"]["dNdE"]
bk_sim = analytical_final_ion_ke_result["bk_sim_density"]
bk_profiles = analytical_final_ion_ke_result["bk_profiles"]

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------
plt.figure(figsize=(7.6, 4.8))

# Analytical curves
plt.plot(
    E_mid,
    normalize_max(y_saalmann),
    "k--",
    lw=2.0,
    label="Saalmann uniform",
)

plt.plot(
    bk_sim["E_mid"],
    normalize_max(bk_sim["dNdE"]),
    lw=2.8,
    label="BK from simulation initial density",
)

for prof in bk_profiles:
    plt.plot(
        prof["E_mid"],
        normalize_max(prof["dNdE"]),
        lw=1.5,
        alpha=0.75,
        label=prof["label"],
    )

# Experimental data
plt.plot(
    E_centers,
    normalize_max(dNdE_exp),
    "o",
    ms=4,
    alpha=0.75,
    label=f"Experiment run {RUN_EXP_ION}",
)

plt.xlabel("Ion kinetic energy (eV)")
plt.ylabel("Normalized signal")
plt.title("Analytical final ion KE distributions + experiment")
plt.xlim(0, E_XMAX_EV)
plt.ylim(0, 1.08)
plt.grid(True, alpha=0.25)
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# =========================================================
# Add Saalmann combined model:
# laser Gaussian profile + Gaussian cluster-size distribution
#
# Run after:
#   analytical_final_ion_ke_result
#
# This cell overlays:
#   - Experiment
#   - BK from simulation initial density
#   - Saalmann single uniform sphere
#   - Saalmann combined laser profile + size distribution
#
# Model:
#   single cluster:
#       dP/dE = (3/2Emax) sqrt(E/Emax), 0 < E < Emax
#
#   Emax(N,I) = Emax_ref * (N/N0)^(2/3) * (I/I0)^gamma
#
#   Then average over:
#       I(rho) = I0 exp(-2 rho^2 / w0^2)
#       P(N)   = Gaussian centered at N0
#
# All final spectra are normalized for plotting.
# =========================================================

import h5py
import numpy as np
import matplotlib.pyplot as plt
import scipy.constants as sc

# ---------------------------------------------------------
# USER INPUTS
# ---------------------------------------------------------
EXP_ION_H5 = EXP_ION_H5
RUN_EXP_ION = 236

# Final energy scale from your simulation/extrapolation
MEAN_FINAL_ENERGY_EV = analytical_final_ion_ke_result["MEAN_FINAL_ENERGY_EV"]

# Saalmann combined model settings
LASER_WAIST_UM = 25.0          # Gaussian beam waist
I0_WCM2 = 1.0e15               # paper/peak intensity; only relative scaling matters here
I_MIN_REL = 5e-2               # integrate laser profile down to I/I0 = 1e-3

N0_CLUSTER = 2.0e6             # Gaussian size distribution center
SIGMA_N_REL = 0.33             # width: sigma_N = SIGMA_N_REL * N0
N_SIGMA_RANGE = 4.0            # truncate Gaussian at ±4 sigma

# Energy scaling exponent:
# For Coulomb explosion, Emax ~ q^2 N/R ~ q^2 N^(2/3).
# If q is roughly proportional to field F, then q^2 ~ I, so gamma=1.
GAMMA_INTENSITY = 1.0

# Grid resolution for convolution
N_SIZE_GRID = 350
N_LASER_GRID = 450

# Plot settings
E_XMAX_EV = 100
SHOW_ONLY_MAIN_CURVES = True   # True = less crowded
USE_LOGY = False

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def trapz_compat(y, x):
    if hasattr(np, "trapezoid"):
        return np.trapezoid(y, x)
    return np.trapz(y, x)

def normalize_max(y):
    y = np.asarray(y, dtype=float)
    m = np.nanmax(y)
    return y / m if np.isfinite(m) and m > 0 else y

def normalize_area(y, x):
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    area = trapz_compat(np.clip(y, 0, None), x)
    return y / area if np.isfinite(area) and area > 0 else y

def centers_to_edges(x):
    x = np.asarray(x, dtype=float)
    dx = np.diff(x)
    edges = np.empty(len(x) + 1, dtype=float)
    edges[1:-1] = 0.5 * (x[:-1] + x[1:])
    edges[0] = x[0] - 0.5 * dx[0]
    edges[-1] = x[-1] + 0.5 * dx[-1]
    return edges

P_AU_SI = sc.physical_constants["atomic unit of momentum"][0]
M_ION_KG = 4.0 * sc.atomic_mass

def p_au_to_E_eV(p_au):
    p_si = np.asarray(p_au, dtype=float) * P_AU_SI
    return p_si**2 / (2.0 * M_ION_KG) / sc.e

def single_cluster_kedi(E, Emax):
    """
    Saalmann ideal homogeneous single-cluster KEDI.
    Area-normalized approximately by construction:
      dP/dE = (3/2Emax) sqrt(E/Emax), 0<E<Emax
    """
    E = np.asarray(E, dtype=float)
    Emax = np.maximum(float(Emax), 1e-30)
    eps = np.clip(E / Emax, 0.0, None)
    y = (1.5 / Emax) * np.sqrt(eps) * (eps <= 1.0)
    return y

# ---------------------------------------------------------
# Read experimental ion distribution in energy space
# ---------------------------------------------------------
def try_read_direct_energy_hist(hist):
    candidates = [
        ("edges_E_eV",  "centers_E_eV",  "counts_E"),
        ("edges_E",     "centers_E",     "counts_E"),
        ("edges_KE_eV", "centers_KE_eV", "counts_KE"),
        ("edges_KE",    "centers_KE",    "counts_KE"),
        ("E_edges",     "E_centers",     "counts_E"),
        ("energy_edges","energy_centers","counts_E"),
    ]

    for edge_key, center_key, count_key in candidates:
        if center_key in hist and count_key in hist:
            E_centers = hist[center_key][:].astype(float)
            counts_E = hist[count_key][:].astype(float)

            if edge_key in hist:
                E_edges = hist[edge_key][:].astype(float)
            else:
                E_edges = centers_to_edges(E_centers)

            dE = np.diff(E_edges)
            dE[dE <= 0] = np.nan
            dNdE = counts_E / dE

            return E_centers, dNdE, f"direct energy histogram: {center_key}, {count_key}"

    return None, None, None

with h5py.File(EXP_ION_H5, "r") as h5:
    hist = h5[f"runs/run_{RUN_EXP_ION}/hist"]

    E_exp, dNdE_exp, exp_source = try_read_direct_energy_hist(hist)

    if E_exp is None:
        if "centers_P" not in hist or "counts_P" not in hist:
            raise KeyError("Need either energy histogram or momentum histogram centers_P/counts_P.")

        p_centers = hist["centers_P"][:].astype(float)
        counts_P = hist["counts_P"][:].astype(float)

        if "edges_P" in hist:
            p_edges = hist["edges_P"][:].astype(float)
        else:
            p_edges = centers_to_edges(p_centers)

        p_edges = np.clip(p_edges, 0.0, None)

        E_edges_exp = p_au_to_E_eV(p_edges)
        E_exp = 0.5 * (E_edges_exp[1:] + E_edges_exp[:-1])

        dE = np.diff(E_edges_exp)
        dE[dE <= 0] = np.nan

        dNdE_exp = counts_P / dE
        exp_source = "converted from momentum histogram using Jacobian"

print(f"Experimental source used: {exp_source}")

# ---------------------------------------------------------
# Build Saalmann combined laser + Gaussian size distribution
# ---------------------------------------------------------
E_mid = analytical_final_ion_ke_result["E_mid"]

# ---- cluster-size distribution: Gaussian centered at 2e6 atoms
sigma_N = SIGMA_N_REL * N0_CLUSTER

N_min = max(1.0, N0_CLUSTER - N_SIGMA_RANGE * sigma_N)
N_max = N0_CLUSTER + N_SIGMA_RANGE * sigma_N

N_grid = np.linspace(N_min, N_max, N_SIZE_GRID)
w_N = np.exp(-0.5 * ((N_grid - N0_CLUSTER) / sigma_N)**2)
w_N /= np.sum(w_N)

# ---- laser profile: Gaussian transverse beam
# I(rho) = I0 exp(-2 rho^2 / w0^2)
w0_um = LASER_WAIST_UM

rho_max_um = w0_um * np.sqrt(-0.5 * np.log(I_MIN_REL))
rho_edges_um = np.linspace(0.0, rho_max_um, N_LASER_GRID + 1)
rho_mid_um = 0.5 * (rho_edges_um[1:] + rho_edges_um[:-1])
drho_um = np.diff(rho_edges_um)

I_rel = np.exp(-2.0 * (rho_mid_um / w0_um)**2)

# clusters uniformly sample area element 2π rho d rho
w_I = rho_mid_um * drho_um
w_I /= np.sum(w_I)

# ---- combined Emax factors
# Emax = Emax_ref * factor
factor_N = (N_grid / N0_CLUSTER)**(2.0 / 3.0)
factor_I = I_rel**GAMMA_INTENSITY

factor_2d = factor_N[:, None] * factor_I[None, :]
weight_2d = w_N[:, None] * w_I[None, :]

# The mean energy of single homogeneous cluster is <E> = 3/5 Emax.
# Therefore the mixture mean is:
#   <E> = (3/5) Emax_ref * <factor>
mean_factor = np.sum(weight_2d * factor_2d)
Emax_ref = MEAN_FINAL_ENERGY_EV / ((3.0 / 5.0) * mean_factor)

print("\nSaalmann combined model:")
print(f"  laser waist w0 = {LASER_WAIST_UM:.2f} um")
print(f"  I0 = {I0_WCM2:.3e} W/cm^2")
print(f"  intensity cutoff I/I0 = {I_MIN_REL:.1e}")
print(f"  cluster N0 = {N0_CLUSTER:.3e}")
print(f"  sigma_N = {sigma_N:.3e} atoms ({SIGMA_N_REL:.2f} relative)")
print(f"  gamma intensity exponent = {GAMMA_INTENSITY:.2f}")
print(f"  mean Emax factor = {mean_factor:.5g}")
print(f"  Emax_ref at N0 and I0 = {Emax_ref:.5g} eV")

# Numerical convolution over N and I
y_saalmann_combined = np.zeros_like(E_mid, dtype=float)

for iN in range(N_SIZE_GRID):
    for iI in range(N_LASER_GRID):
        Emax_local = Emax_ref * factor_2d[iN, iI]
        y_saalmann_combined += weight_2d[iN, iI] * single_cluster_kedi(E_mid, Emax_local)

y_saalmann_combined = normalize_area(y_saalmann_combined, E_mid)

mean_check = trapz_compat(E_mid * y_saalmann_combined, E_mid)
print(f"  mean energy check from histogram = {mean_check:.4f} eV")

# ---------------------------------------------------------
# Get older analytical curves from previous cell
# ---------------------------------------------------------
y_saalmann_single = analytical_final_ion_ke_result["saalmann"]["dNdE"]
bk_sim = analytical_final_ion_ke_result["bk_sim_density"]
bk_profiles = analytical_final_ion_ke_result["bk_profiles"]

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------
plt.figure(figsize=(7.8, 4.9))

# Experiment
plt.plot(
    E_exp,
    normalize_max(dNdE_exp),
    "o",
    ms=4,
    alpha=0.72,
    label=f"Experiment run {RUN_EXP_ION}",
)

plt.plot(
    E_mid,
    normalize_max(y_saalmann_combined),
    lw=2.9,
    label="Saalmann laser profile + Gaussian N",
)

# BK from simulation initial density
plt.plot(
    bk_sim["E_mid"],
    normalize_max(bk_sim["dNdE"]),
    lw=2.4,
    label="BK from simulation initial density",
)

# Optional extra BK profiles
if not SHOW_ONLY_MAIN_CURVES:
    for prof in bk_profiles:
        plt.plot(
            prof["E_mid"],
            normalize_max(prof["dNdE"]),
            lw=1.3,
            alpha=0.55,
            label=prof["label"],
        )

plt.xlabel("Ion kinetic energy (eV)")
plt.ylabel("Normalized signal")
plt.title("Experiment vs Saalmann combined model and BK model")
plt.xlim(0, E_XMAX_EV)

if USE_LOGY:
    plt.yscale("log")
    plt.ylim(1e-4, 2.0)
else:
    plt.ylim(0, 1.08)

plt.grid(True, alpha=0.25)
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# Save result
# ---------------------------------------------------------
saalmann_combined_result = {
    "E_mid": E_mid,
    "dNdE": y_saalmann_combined,
    "MEAN_FINAL_ENERGY_EV": MEAN_FINAL_ENERGY_EV,
    "Emax_ref_eV": Emax_ref,
    "LASER_WAIST_UM": LASER_WAIST_UM,
    "I0_WCM2": I0_WCM2,
    "I_MIN_REL": I_MIN_REL,
    "N0_CLUSTER": N0_CLUSTER,
    "SIGMA_N_REL": SIGMA_N_REL,
    "sigma_N": sigma_N,
    "GAMMA_INTENSITY": GAMMA_INTENSITY,
    "N_grid": N_grid,
    "w_N": w_N,
    "rho_mid_um": rho_mid_um,
    "I_rel": I_rel,
    "w_I": w_I,
    "mean_check_eV": mean_check,
    "E_exp": E_exp,
    "dNdE_exp": dNdE_exp,
    "exp_source": exp_source,
}

print("\nSaved dictionary: saalmann_combined_result")


In [ ]:

# =========================================================
# Fit Saalmann combined model to experimental ion KEDI
#
# Model:
#   single cluster:
#       dP/dE = (3/2Emax) sqrt(E/Emax), 0 < E < Emax
#
#   Emax(N,I) = Emax_ref * (N/N0)^(2/3) * (I/I0)^gamma
#
# Average over:
#   laser Gaussian transverse profile
#   Gaussian cluster-size distribution centered at N0
#
# Fit parameters:
#   MEAN_FINAL_ENERGY_EV
#   SIGMA_N_REL
#   I_MIN_REL
#   optionally GAMMA_INTENSITY
#
# Important:
#   N0 is kept fixed because, once the energy scale is allowed
#   to float, normalized KEDI shape mostly constrains the width
#   of the size distribution, not the absolute center N0.
# =========================================================

import h5py
import numpy as np
import matplotlib.pyplot as plt
import scipy.constants as sc

from scipy.optimize import differential_evolution, minimize

# ---------------------------------------------------------
# USER INPUTS
# ---------------------------------------------------------
EXP_ION_H5 = EXP_ION_H5
RUN_EXP_ION = 236

# Fixed experimental / physical parameters
LASER_WAIST_UM = 25.0
I0_WCM2 = 1.0e15

N0_CLUSTER = 2.0e6

# Fit region
FIT_E_MIN_EV = 1.0
FIT_E_MAX_EV = 100.0

# Plot range
E_XMAX_EV = 100.0

# Numerical grids
N_SIZE_GRID = 180
N_LASER_GRID = 220

# Fit controls
FIT_GAMMA = False       # first keep gamma fixed to 1.0
GAMMA_FIXED = 1.0

N_REPEAT = 8            # repeated global fits with different random seeds

# Parameter bounds
# If FIT_GAMMA=False:
#   params = [Emean_eV, sigma_N_rel, I_min_rel]
#
# If FIT_GAMMA=True:
#   params = [Emean_eV, sigma_N_rel, I_min_rel, gamma]
BOUNDS_NO_GAMMA = [
    (15.0, 120.0),      # Emean_eV
    (0.05, 0.90),       # sigma_N_rel
    (0.005, 0.30),      # I_min_rel
]

BOUNDS_WITH_GAMMA = [
    (15.0, 120.0),      # Emean_eV
    (0.05, 0.90),       # sigma_N_rel
    (0.005, 0.30),      # I_min_rel
    (0.50, 2.00),       # gamma
]

# Starting reference values, just for plotting/initial intuition
P0_REFERENCE = {
    "Emean_eV": 51.0,
    "sigma_N_rel": 0.33,
    "I_min_rel": 5e-2,
    "gamma": 1.0,
}

# Objective weights
# linear term controls main peak/shape
# log term controls tail without letting zeros explode
LOG_WEIGHT = 0.15

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def trapz_compat(y, x):
    if hasattr(np, "trapezoid"):
        return np.trapezoid(y, x)
    return np.trapz(y, x)

def normalize_area(y, x):
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    area = trapz_compat(np.clip(y, 0, None), x)
    return y / area if np.isfinite(area) and area > 0 else y

def normalize_max_in_mask(y, mask):
    y = np.asarray(y, dtype=float)
    m = np.nanmax(y[mask])
    return y / m if np.isfinite(m) and m > 0 else y

def centers_to_edges(x):
    x = np.asarray(x, dtype=float)
    dx = np.diff(x)
    edges = np.empty(len(x) + 1, dtype=float)
    edges[1:-1] = 0.5 * (x[:-1] + x[1:])
    edges[0] = x[0] - 0.5 * dx[0]
    edges[-1] = x[-1] + 0.5 * dx[-1]
    return edges

P_AU_SI = sc.physical_constants["atomic unit of momentum"][0]
M_ION_KG = 4.0 * sc.atomic_mass

def p_au_to_E_eV(p_au):
    p_si = np.asarray(p_au, dtype=float) * P_AU_SI
    return p_si**2 / (2.0 * M_ION_KG) / sc.e

def single_cluster_kedi(E, Emax):
    E = np.asarray(E, dtype=float)
    Emax = np.maximum(Emax, 1e-30)

    eps = np.clip(E[..., None] / Emax[None, :], 0.0, None)
    y = (1.5 / Emax[None, :]) * np.sqrt(eps) * (eps <= 1.0)
    return y

def read_exp_energy_distribution(h5_path, run_no):
    with h5py.File(h5_path, "r") as h5:
        hist = h5[f"runs/run_{run_no}/hist"]

        print("Available experimental hist keys:")
        for k in hist.keys():
            print("  ", k)

        candidates = [
            ("edges_E_eV",  "centers_E_eV",  "counts_E"),
            ("edges_E",     "centers_E",     "counts_E"),
            ("edges_KE_eV", "centers_KE_eV", "counts_KE"),
            ("edges_KE",    "centers_KE",    "counts_KE"),
            ("E_edges",     "E_centers",     "counts_E"),
            ("energy_edges","energy_centers","counts_E"),
        ]

        for edge_key, center_key, count_key in candidates:
            if center_key in hist and count_key in hist:
                E_centers = hist[center_key][:].astype(float)
                counts_E = hist[count_key][:].astype(float)

                if edge_key in hist:
                    E_edges = hist[edge_key][:].astype(float)
                else:
                    E_edges = centers_to_edges(E_centers)

                dE = np.diff(E_edges)
                dE[dE <= 0] = np.nan
                dNdE = counts_E / dE

                return E_centers, dNdE, f"direct energy histogram: {center_key}, {count_key}"

        if "centers_P" not in hist or "counts_P" not in hist:
            raise KeyError("Need either energy histogram or momentum histogram centers_P/counts_P.")

        p_centers = hist["centers_P"][:].astype(float)
        counts_P = hist["counts_P"][:].astype(float)

        if "edges_P" in hist:
            p_edges = hist["edges_P"][:].astype(float)
        else:
            p_edges = centers_to_edges(p_centers)

        p_edges = np.clip(p_edges, 0.0, None)

        E_edges = p_au_to_E_eV(p_edges)
        E_centers = 0.5 * (E_edges[1:] + E_edges[:-1])

        dE = np.diff(E_edges)
        dE[dE <= 0] = np.nan

        dNdE = counts_P / dE

        return E_centers, dNdE, "converted from momentum histogram using Jacobian"

# ---------------------------------------------------------
# Saalmann combined model
# ---------------------------------------------------------
def saalmann_combined_kedi(
    E_grid,
    Emean_eV,
    sigma_N_rel,
    I_min_rel,
    gamma=1.0,
    n_size_grid=N_SIZE_GRID,
    n_laser_grid=N_LASER_GRID,
):
    """
    Saalmann combined KEDI:
      Gaussian size distribution centered at N0, expressed in x=N/N0.
      Gaussian laser profile integrated from I_min/I0 to 1.

    The 25 um waist cancels out in normalized shape, except for
    translating I_min/I0 into a physical radius cutoff.
    """

    E_grid = np.asarray(E_grid, dtype=float)

    sigma_N_rel = float(np.clip(sigma_N_rel, 1e-4, 5.0))
    I_min_rel = float(np.clip(I_min_rel, 1e-6, 0.999))
    gamma = float(gamma)

    # ---- size distribution in x = N/N0
    # Use a fixed x grid so N0 remains a center/reference.
    x_min = max(1e-4, 1.0 - 5.0 * sigma_N_rel)
    x_max = 1.0 + 5.0 * sigma_N_rel

    x_N = np.linspace(x_min, x_max, n_size_grid)
    dx_N = x_N[1] - x_N[0]

    w_N = np.exp(-0.5 * ((x_N - 1.0) / sigma_N_rel)**2)
    w_N *= dx_N
    w_N /= np.sum(w_N)

    factor_N = x_N**(2.0 / 3.0)

    # ---- laser profile
    # For Gaussian I/I0 = exp(-2 rho^2/w0^2).
    # Area element gives uniform weighting in ln(I/I0):
    #   dA ∝ d ln(1/u), u=I/I0.
    u_edges = np.geomspace(I_min_rel, 1.0, n_laser_grid + 1)
    u_mid = np.sqrt(u_edges[:-1] * u_edges[1:])

    # equal area weight in log-intensity bins
    w_I = np.diff(np.log(u_edges))
    w_I /= np.sum(w_I)

    factor_I = u_mid**gamma

    # ---- combined factors and weights
    factor = (factor_N[:, None] * factor_I[None, :]).ravel()
    weight = (w_N[:, None] * w_I[None, :]).ravel()

    # Mean energy of each single-cluster distribution is (3/5) Emax.
    # We choose Emax_ref so the mixture mean is Emean_eV.
    mean_factor = np.sum(weight * factor)
    Emax_ref = Emean_eV / ((3.0 / 5.0) * mean_factor)

    Emax_all = Emax_ref * factor

    Y_components = single_cluster_kedi(E_grid, Emax_all)
    y = np.sum(Y_components * weight[None, :], axis=1)

    y = normalize_area(y, E_grid)

    return y, {
        "Emax_ref_eV": Emax_ref,
        "mean_factor": mean_factor,
        "x_N": x_N,
        "w_N": w_N,
        "u_mid": u_mid,
        "w_I": w_I,
        "rho_max_um": LASER_WAIST_UM * np.sqrt(-0.5 * np.log(I_min_rel)),
        "I_min_abs_Wcm2": I_min_rel * I0_WCM2,
    }

# ---------------------------------------------------------
# Read and prepare experimental data
# ---------------------------------------------------------
E_exp_raw, dNdE_exp_raw, exp_source = read_exp_energy_distribution(EXP_ION_H5, RUN_EXP_ION)

m_clean = (
    np.isfinite(E_exp_raw)
    & np.isfinite(dNdE_exp_raw)
    & (E_exp_raw >= 0)
    & (dNdE_exp_raw >= 0)
)

E_exp = E_exp_raw[m_clean]
dNdE_exp = dNdE_exp_raw[m_clean]

order = np.argsort(E_exp)
E_exp = E_exp[order]
dNdE_exp = dNdE_exp[order]

fit_mask = (
    (E_exp >= FIT_E_MIN_EV)
    & (E_exp <= FIT_E_MAX_EV)
    & np.isfinite(dNdE_exp)
    & (dNdE_exp > 0)
)

if np.sum(fit_mask) < 10:
    raise RuntimeError("Too few experimental points in fit range.")

y_exp_norm = normalize_max_in_mask(dNdE_exp, fit_mask)

print(f"\nExperimental source: {exp_source}")
print(f"Fit range: {FIT_E_MIN_EV:.2f} to {FIT_E_MAX_EV:.2f} eV")
print(f"Fit points: {np.sum(fit_mask)}")

# ---------------------------------------------------------
# Objective
# ---------------------------------------------------------
def unpack_params(p):
    if FIT_GAMMA:
        Emean_eV, sigma_N_rel, I_min_rel, gamma = p
    else:
        Emean_eV, sigma_N_rel, I_min_rel = p
        gamma = GAMMA_FIXED

    return Emean_eV, sigma_N_rel, I_min_rel, gamma

def objective(p):
    Emean_eV, sigma_N_rel, I_min_rel, gamma = unpack_params(p)

    y_model, _ = saalmann_combined_kedi(
        E_exp,
        Emean_eV=Emean_eV,
        sigma_N_rel=sigma_N_rel,
        I_min_rel=I_min_rel,
        gamma=gamma,
    )

    y_model_norm = normalize_max_in_mask(y_model, fit_mask)

    m = fit_mask & np.isfinite(y_model_norm) & (y_model_norm > 0)

    if np.sum(m) < 10:
        return 1e30

    # Linear residual
    r_lin = y_model_norm[m] - y_exp_norm[m]

    # Log residual, protected near zero
    floor = 1e-4
    r_log = (
        np.log10(np.maximum(y_model_norm[m], floor))
        - np.log10(np.maximum(y_exp_norm[m], floor))
    )

    return np.mean(r_lin**2) + LOG_WEIGHT * np.mean(r_log**2)

# ---------------------------------------------------------
# Repeated global + local optimization
# ---------------------------------------------------------
bounds = BOUNDS_WITH_GAMMA if FIT_GAMMA else BOUNDS_NO_GAMMA

all_results = []

for seed in range(N_REPEAT):
    res_de = differential_evolution(
        objective,
        bounds=bounds,
        seed=1000 + seed,
        polish=False,
        maxiter=80,
        popsize=12,
        tol=1e-5,
        updating="immediate",
        workers=1,
    )

    res_local = minimize(
        objective,
        res_de.x,
        method="Nelder-Mead",
        options={
            "maxiter": 3000,
            "xatol": 1e-5,
            "fatol": 1e-7,
        },
    )

    p_best = res_local.x if res_local.fun < res_de.fun else res_de.x
    loss_best = min(res_local.fun, res_de.fun)

    all_results.append({
        "seed": seed,
        "params": p_best,
        "loss": loss_best,
        "de_loss": res_de.fun,
        "local_loss": res_local.fun,
    })

    Emean_eV, sigma_N_rel, I_min_rel, gamma = unpack_params(p_best)

    print(
        f"repeat {seed:02d}: loss={loss_best:.5g} | "
        f"Emean={Emean_eV:.3f} eV, "
        f"sigma_N_rel={sigma_N_rel:.3f}, "
        f"I_min/I0={I_min_rel:.4f}, "
        f"gamma={gamma:.3f}"
    )

best = min(all_results, key=lambda d: d["loss"])
p_best = best["params"]

Emean_best, sigma_N_best, Imin_best, gamma_best = unpack_params(p_best)

y_best, info_best = saalmann_combined_kedi(
    E_exp,
    Emean_eV=Emean_best,
    sigma_N_rel=sigma_N_best,
    I_min_rel=Imin_best,
    gamma=gamma_best,
)

y_best_norm = normalize_max_in_mask(y_best, fit_mask)

print("\n================ BEST SAALMANN FIT ================")
print(f"loss                  = {best['loss']:.6g}")
print(f"MEAN_FINAL_ENERGY_EV  = {Emean_best:.6g}")
print(f"SIGMA_N_REL           = {sigma_N_best:.6g}")
print(f"I_MIN_REL             = {Imin_best:.6g}")
print(f"GAMMA_INTENSITY       = {gamma_best:.6g}")
print(f"Emax_ref_eV           = {info_best['Emax_ref_eV']:.6g}")
print(f"I_min_abs             = {info_best['I_min_abs_Wcm2']:.3e} W/cm^2")
print(f"rho_max               = {info_best['rho_max_um']:.3f} um for w0={LASER_WAIST_UM:.1f} um")
print("====================================================")

# ---------------------------------------------------------
# Reference curve from your manual values
# ---------------------------------------------------------
y_ref, info_ref = saalmann_combined_kedi(
    E_exp,
    Emean_eV=P0_REFERENCE["Emean_eV"],
    sigma_N_rel=P0_REFERENCE["sigma_N_rel"],
    I_min_rel=P0_REFERENCE["I_min_rel"],
    gamma=P0_REFERENCE["gamma"],
)

y_ref_norm = normalize_max_in_mask(y_ref, fit_mask)

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.8))

# Linear plot
ax = axes[0]

ax.plot(
    E_exp,
    y_exp_norm,
    "o",
    ms=4,
    alpha=0.65,
    label=f"Experiment run {RUN_EXP_ION}",
)

ax.plot(
    E_exp,
    y_ref_norm,
    lw=1.8,
    ls="--",
    alpha=0.8,
    label=(
        "manual ref: "
        rf"$\sigma_N/N_0={P0_REFERENCE['sigma_N_rel']:.2f}$, "
        rf"$I_{{min}}/I_0={P0_REFERENCE['I_min_rel']:.2g}$"
    ),
)

ax.plot(
    E_exp,
    y_best_norm,
    lw=2.8,
    label=(
        "best fit: "
        rf"$\sigma_N/N_0={sigma_N_best:.2f}$, "
        rf"$I_{{min}}/I_0={Imin_best:.3g}$"
    ),
)

ax.axvspan(FIT_E_MIN_EV, FIT_E_MAX_EV, color="gray", alpha=0.08, label="fit range")

ax.set_xlabel("Ion kinetic energy (eV)")
ax.set_ylabel("Normalized signal")
ax.set_xlim(0, E_XMAX_EV)
ax.set_ylim(0, 1.08)
ax.set_title("Saalmann combined model fit")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

# Log plot
ax = axes[1]

ax.plot(
    E_exp,
    np.maximum(y_exp_norm, 1e-6),
    "o",
    ms=4,
    alpha=0.65,
    label="Experiment",
)

ax.plot(
    E_exp,
    np.maximum(y_ref_norm, 1e-6),
    lw=1.8,
    ls="--",
    alpha=0.8,
    label="manual reference",
)

ax.plot(
    E_exp,
    np.maximum(y_best_norm, 1e-6),
    lw=2.8,
    label="best fit",
)

ax.axvspan(FIT_E_MIN_EV, FIT_E_MAX_EV, color="gray", alpha=0.08)

ax.set_xlabel("Ion kinetic energy (eV)")
ax.set_ylabel("Normalized signal")
ax.set_xlim(0, E_XMAX_EV)
ax.set_yscale("log")
ax.set_ylim(1e-4, 2)
ax.set_title("Same comparison, log scale")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# Save result
# ---------------------------------------------------------
saalmann_fit_to_exp_result = {
    "best": best,
    "all_results": all_results,
    "E_exp": E_exp,
    "dNdE_exp": dNdE_exp,
    "y_exp_norm": y_exp_norm,
    "fit_mask": fit_mask,
    "y_best_norm": y_best_norm,
    "y_ref_norm": y_ref_norm,
    "Emean_best_eV": Emean_best,
    "sigma_N_rel_best": sigma_N_best,
    "I_min_rel_best": Imin_best,
    "gamma_best": gamma_best,
    "info_best": info_best,
    "EXP_ION_H5": EXP_ION_H5,
    "RUN_EXP_ION": RUN_EXP_ION,
    "N0_CLUSTER": N0_CLUSTER,
    "LASER_WAIST_UM": LASER_WAIST_UM,
    "I0_WCM2": I0_WCM2,
    "FIT_E_MIN_EV": FIT_E_MIN_EV,
    "FIT_E_MAX_EV": FIT_E_MAX_EV,
}

print("\nSaved dictionary: saalmann_fit_to_exp_result")


In [ ]:
# =========================================================
# Manual Saalmann combined model
# NO FITTING
#
# Same logic as the working cell:
#   - clean uniform E grid
#   - laser Gaussian profile
#   - cluster-size distribution
#   - manual parameters only
#
# You can choose:
#   SIZE_DISTRIBUTION_MODE = "lognormal_source"   # fixed source distribution
#   SIZE_DISTRIBUTION_MODE = "gaussian_old"       # reproduces old working cell
# =========================================================

import h5py
import numpy as np
import matplotlib.pyplot as plt
import scipy.constants as sc

# ---------------------------------------------------------
# USER INPUTS
# ---------------------------------------------------------
EXP_ION_H5 = EXP_ION_H5
RUN_EXP_ION = 236

# ---------------------------------------------------------
# MANUAL SAALMANN PARAMETERS
# ---------------------------------------------------------
MEAN_FINAL_ENERGY_EV = 51.0      # try 15.0 from optimizer, or 51.0 from simulation
I_MIN_REL = 0.01             # best early fit was around 0.0619
GAMMA_INTENSITY = 1.0

# ---------------------------------------------------------
# FIXED SOURCE PARAMETERS
# ---------------------------------------------------------
NOZZLE_DIAMETER_UM = 5.0
NOZZLE_T_K = 9.0
STAGNATION_PRESSURE_BAR = 50.0

N0_CLUSTER = 2.0e6

# Choose one:
SIZE_DISTRIBUTION_MODE = "lognormal_source"
# SIZE_DISTRIBUTION_MODE = "gaussian_old"

# For lognormal source distribution
NU_SIZE_FIXED = 0.33

# For old Gaussian-in-N distribution
SIGMA_N_REL_OLD = 0.33

N_SIGMA_RANGE = 4.0

# ---------------------------------------------------------
# Laser parameters
# ---------------------------------------------------------
LASER_WAIST_UM = 25.0
I0_WCM2 = 1.0e15

# ---------------------------------------------------------
# Numerical grids
# ---------------------------------------------------------
N_SIZE_GRID = 350
N_LASER_GRID = 450

E_MAX_EV = 100.0
N_E_BINS = 600

USE_LOGY = False

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def trapz_compat(y, x):
    if hasattr(np, "trapezoid"):
        return np.trapezoid(y, x)
    return np.trapz(y, x)

def normalize_area(y, x):
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    area = trapz_compat(np.clip(y, 0, None), x)
    return y / area if np.isfinite(area) and area > 0 else y

def normalize_max(y):
    y = np.asarray(y, dtype=float)
    m = np.nanmax(y)
    return y / m if np.isfinite(m) and m > 0 else y

def centers_to_edges(x):
    x = np.asarray(x, dtype=float)
    dx = np.diff(x)
    edges = np.empty(len(x) + 1)
    edges[1:-1] = 0.5 * (x[:-1] + x[1:])
    edges[0] = x[0] - 0.5 * dx[0]
    edges[-1] = x[-1] + 0.5 * dx[-1]
    return edges

P_AU_SI = sc.physical_constants["atomic unit of momentum"][0]
M_ION_KG = 4.0 * sc.atomic_mass

def p_au_to_E_eV(p_au):
    p_si = np.asarray(p_au, dtype=float) * P_AU_SI
    return p_si**2 / (2.0 * M_ION_KG) / sc.e

def single_cluster_kedi(E, Emax):
    """
    Saalmann ideal homogeneous single-cluster KEDI:
        dP/dE = (3/2Emax) sqrt(E/Emax), 0 < E < Emax
    """
    E = np.asarray(E, dtype=float)
    Emax = np.asarray(Emax, dtype=float)
    Emax = np.maximum(Emax, 1e-30)

    eps = np.clip(E[:, None] / Emax[None, :], 0.0, None)
    y = (1.5 / Emax[None, :]) * np.sqrt(eps) * (eps <= 1.0)
    return y

def mixture_saalmann(E_grid, Emax_all, weight_all, chunk=2000):
    """
    Chunked vectorized convolution over all Emax values.
    Faster than a huge double loop and avoids very large memory use.
    """
    y = np.zeros_like(E_grid, dtype=float)

    n = len(Emax_all)
    for i0 in range(0, n, chunk):
        i1 = min(i0 + chunk, n)
        Y = single_cluster_kedi(E_grid, Emax_all[i0:i1])
        y += np.sum(Y * weight_all[i0:i1][None, :], axis=1)

    return normalize_area(y, E_grid)

def read_exp_energy_distribution(h5_path, run_no):
    with h5py.File(h5_path, "r") as h5:
        hist = h5[f"runs/run_{run_no}/hist"]

        candidates = [
            ("edges_E_eV",  "centers_E_eV",  "counts_E"),
            ("edges_E",     "centers_E",     "counts_E"),
            ("edges_KE_eV", "centers_KE_eV", "counts_KE"),
            ("edges_KE",    "centers_KE",    "counts_KE"),
            ("E_edges",     "E_centers",     "counts_E"),
            ("energy_edges","energy_centers","counts_E"),
        ]

        for edge_key, center_key, count_key in candidates:
            if center_key in hist and count_key in hist:
                E_centers = hist[center_key][:].astype(float)
                counts_E = hist[count_key][:].astype(float)

                if edge_key in hist:
                    E_edges = hist[edge_key][:].astype(float)
                else:
                    E_edges = centers_to_edges(E_centers)

                dE = np.diff(E_edges)
                dE[dE <= 0] = np.nan
                dNdE = counts_E / dE

                return E_centers, dNdE, f"direct energy histogram: {center_key}, {count_key}"

        if "centers_P" not in hist or "counts_P" not in hist:
            raise KeyError("Need either energy histogram or momentum histogram centers_P/counts_P.")

        p_centers = hist["centers_P"][:].astype(float)
        counts_P = hist["counts_P"][:].astype(float)

        if "edges_P" in hist:
            p_edges = hist["edges_P"][:].astype(float)
        else:
            p_edges = centers_to_edges(p_centers)

        p_edges = np.clip(p_edges, 0.0, None)

        E_edges = p_au_to_E_eV(p_edges)
        E_centers = 0.5 * (E_edges[1:] + E_edges[:-1])

        dE = np.diff(E_edges)
        dE[dE <= 0] = np.nan

        dNdE = counts_P / dE

        return E_centers, dNdE, "converted from momentum histogram using Jacobian"

# ---------------------------------------------------------
# Experimental data
# ---------------------------------------------------------
E_exp, dNdE_exp, exp_source = read_exp_energy_distribution(EXP_ION_H5, RUN_EXP_ION)

m_exp = (
    np.isfinite(E_exp)
    & np.isfinite(dNdE_exp)
    & (E_exp >= 0)
    & (dNdE_exp >= 0)
)

E_exp = E_exp[m_exp]
dNdE_exp = dNdE_exp[m_exp]

order = np.argsort(E_exp)
E_exp = E_exp[order]
dNdE_exp = dNdE_exp[order]

# ---------------------------------------------------------
# Clean model energy grid
# ---------------------------------------------------------
E_edges = np.linspace(0.0, E_MAX_EV, N_E_BINS + 1)
E_mid = 0.5 * (E_edges[1:] + E_edges[:-1])

# ---------------------------------------------------------
# Cluster-size distribution
# ---------------------------------------------------------
if SIZE_DISTRIBUTION_MODE == "gaussian_old":
    sigma_N = SIGMA_N_REL_OLD * N0_CLUSTER

    N_min = max(1.0, N0_CLUSTER - N_SIGMA_RANGE * sigma_N)
    N_max = N0_CLUSTER + N_SIGMA_RANGE * sigma_N

    N_grid = np.linspace(N_min, N_max, N_SIZE_GRID)

    w_N = np.exp(-0.5 * ((N_grid - N0_CLUSTER) / sigma_N)**2)
    w_N /= np.sum(w_N)

    size_label = rf"Gaussian in $N$, $\sigma_N/N_0={SIGMA_N_REL_OLD:.2f}$"

elif SIZE_DISTRIBUTION_MODE == "lognormal_source":
    lnN0 = np.log(N0_CLUSTER)

    lnN_min = lnN0 - N_SIGMA_RANGE * NU_SIZE_FIXED
    lnN_max = lnN0 + N_SIGMA_RANGE * NU_SIZE_FIXED

    lnN_grid = np.linspace(lnN_min, lnN_max, N_SIZE_GRID)
    N_grid = np.exp(lnN_grid)

    dlnN = lnN_grid[1] - lnN_grid[0]

    w_N = np.exp(-0.5 * ((lnN_grid - lnN0) / NU_SIZE_FIXED)**2) * dlnN
    w_N /= np.sum(w_N)

    size_label = rf"log-normal source, $\nu={NU_SIZE_FIXED:.2f}$"

else:
    raise ValueError("SIZE_DISTRIBUTION_MODE must be 'gaussian_old' or 'lognormal_source'.")

factor_N = (N_grid / N0_CLUSTER)**(2.0 / 3.0)

# ---------------------------------------------------------
# Laser Gaussian transverse profile
# ---------------------------------------------------------
rho_max_um = LASER_WAIST_UM * np.sqrt(-0.5 * np.log(I_MIN_REL))

rho_edges_um = np.linspace(0.0, rho_max_um, N_LASER_GRID + 1)
rho_mid_um = 0.5 * (rho_edges_um[1:] + rho_edges_um[:-1])
drho_um = np.diff(rho_edges_um)

I_rel = np.exp(-2.0 * (rho_mid_um / LASER_WAIST_UM)**2)

# area weighting: dA proportional to rho drho
w_I = rho_mid_um * drho_um
w_I /= np.sum(w_I)

factor_I = I_rel**GAMMA_INTENSITY

# ---------------------------------------------------------
# Combined Saalmann convolution
# ---------------------------------------------------------
factor_2d = factor_N[:, None] * factor_I[None, :]
weight_2d = w_N[:, None] * w_I[None, :]

factor_all = factor_2d.ravel()
weight_all = weight_2d.ravel()

mean_factor = np.sum(weight_all * factor_all)

# <E> = (3/5) Emax_ref <factor>
Emax_ref = MEAN_FINAL_ENERGY_EV / ((3.0 / 5.0) * mean_factor)
Emax_all = Emax_ref * factor_all

y_saalmann = mixture_saalmann(E_mid, Emax_all, weight_all, chunk=2000)
mean_check = trapz_compat(E_mid * y_saalmann, E_mid)

# ---------------------------------------------------------
# Print summary
# ---------------------------------------------------------
print("Manual Saalmann combined model")
print("--------------------------------")
print(f"Experimental source: {exp_source}")
print()
print("Source / size distribution:")
print(f"  nozzle diameter = {NOZZLE_DIAMETER_UM:.1f} um")
print(f"  T0 = {NOZZLE_T_K:.1f} K")
print(f"  P0 = {STAGNATION_PRESSURE_BAR:.1f} bar")
print(f"  N0 fixed = {N0_CLUSTER:.3e}")
print(f"  mode = {SIZE_DISTRIBUTION_MODE}")
print(f"  {size_label}")
print(f"  N range = {N_grid.min():.3e} to {N_grid.max():.3e}")
print()
print("Manual knobs:")
print(f"  mean final energy = {MEAN_FINAL_ENERGY_EV:.4g} eV")
print(f"  I_min/I0 = {I_MIN_REL:.5g}")
print(f"  gamma = {GAMMA_INTENSITY:.3f}")
print()
print("Laser cutoff:")
print(f"  I_min = {I_MIN_REL * I0_WCM2:.3e} W/cm^2")
print(f"  rho_max = {rho_max_um:.3f} um for w0 = {LASER_WAIST_UM:.1f} um")
print()
print("Energy scaling:")
print(f"  mean factor = {mean_factor:.5g}")
print(f"  Emax_ref at N0, I0 = {Emax_ref:.5g} eV")
print(f"  mean energy check = {mean_check:.4g} eV")

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))

# Linear
ax = axes[0]

ax.plot(
    E_exp,
    normalize_max(dNdE_exp),
    "o",
    ms=4,
    alpha=0.70,
    label=f"Experiment run {RUN_EXP_ION}",
)

ax.plot(
    E_mid,
    normalize_max(y_saalmann),
    lw=2.9,
    label=(
        rf"Saalmann: $\langle E\rangle={MEAN_FINAL_ENERGY_EV:.1f}$ eV, "
        rf"$I_{{min}}/I_0={I_MIN_REL:.3g}$"
    ),
)

ax.set_xlabel("Ion kinetic energy (eV)")
ax.set_ylabel("Normalized signal")
ax.set_xlim(0, E_MAX_EV)
ax.set_ylim(0, 1.08)
ax.set_title(f"Manual Saalmann model\n{size_label}")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

# Log
ax = axes[1]

ax.plot(
    E_exp,
    np.maximum(normalize_max(dNdE_exp), 1e-6),
    "o",
    ms=4,
    alpha=0.70,
    label="Experiment",
)

ax.plot(
    E_mid,
    np.maximum(normalize_max(y_saalmann), 1e-6),
    lw=2.9,
    label="Saalmann",
)

ax.set_xlabel("Ion kinetic energy (eV)")
ax.set_ylabel("Normalized signal")
ax.set_xlim(0, E_MAX_EV)
ax.set_yscale("log")
ax.set_ylim(1e-4, 2.0)
ax.set_title("Same comparison, log scale")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# Save result
# ---------------------------------------------------------
saalmann_manual_result = {
    "E_exp": E_exp,
    "dNdE_exp": dNdE_exp,
    "E_mid": E_mid,
    "dNdE_saalmann": y_saalmann,
    "MEAN_FINAL_ENERGY_EV": MEAN_FINAL_ENERGY_EV,
    "I_MIN_REL": I_MIN_REL,
    "GAMMA_INTENSITY": GAMMA_INTENSITY,
    "N0_CLUSTER": N0_CLUSTER,
    "SIZE_DISTRIBUTION_MODE": SIZE_DISTRIBUTION_MODE,
    "NU_SIZE_FIXED": NU_SIZE_FIXED,
    "SIGMA_N_REL_OLD": SIGMA_N_REL_OLD,
    "N_grid": N_grid,
    "w_N": w_N,
    "I_rel": I_rel,
    "w_I": w_I,
    "rho_mid_um": rho_mid_um,
    "rho_max_um": rho_max_um,
    "Emax_ref_eV": Emax_ref,
    "mean_factor": mean_factor,
    "mean_check_eV": mean_check,
    "exp_source": exp_source,
}

print("\nSaved dictionary: saalmann_manual_result")


In [ ]:
# =========================================================
# Saalmann model with:
#   - fixed helium droplet size distribution
#   - Gaussian laser beam profile
#   - manual mean energy scale
#   - three I_min/I0 values for sensitivity
#
# Fixed source conditions:
#   nozzle = 5 um
#   T0 = 9 K
#   P0 = 50 bar
#
# Fixed droplet-size distribution:
#   log-normal, centered at N0 = 2e6 atoms
#   width nu = 0.33
#
# Manual knobs:
#   MEAN_FINAL_ENERGY_EV
#   I_MIN_REL_VALUES
#   GAMMA_INTENSITY
# =========================================================

import h5py
import numpy as np
import matplotlib.pyplot as plt
import scipy.constants as sc

# ---------------------------------------------------------
# USER INPUTS
# ---------------------------------------------------------
EXP_ION_H5 = EXP_ION_H5
RUN_EXP_ION = 236

# Manual Saalmann parameters
MEAN_FINAL_ENERGY_EV = 51.0
GAMMA_INTENSITY = 1.0

# Best one + two comparison values
I_MIN_REL_VALUES = [0.005, 0.01, 0.05]
BEST_I_MIN_REL = 0.01

# Fixed helium source parameters
NOZZLE_DIAMETER_UM = 5.0
NOZZLE_T_K = 9.0
STAGNATION_PRESSURE_BAR = 50.0

# Fixed helium droplet size distribution
N0_CLUSTER = 2.0e6
NU_SIZE_FIXED = 0.33
N_SIGMA_RANGE = 4.0

# Laser
LASER_WAIST_UM = 25.0
I0_WCM2 = 1.0e15

# Numerical grids
N_SIZE_GRID = 350
N_LASER_GRID = 450
E_MAX_EV = 100.0
N_E_BINS = 600

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def trapz_compat(y, x):
    if hasattr(np, "trapezoid"):
        return np.trapezoid(y, x)
    return np.trapz(y, x)

def normalize_area(y, x):
    area = trapz_compat(np.clip(y, 0, None), x)
    return y / area if np.isfinite(area) and area > 0 else y

def normalize_max(y):
    m = np.nanmax(y)
    return y / m if np.isfinite(m) and m > 0 else y

def centers_to_edges(x):
    x = np.asarray(x, dtype=float)
    dx = np.diff(x)
    edges = np.empty(len(x) + 1, dtype=float)
    edges[1:-1] = 0.5 * (x[:-1] + x[1:])
    edges[0] = x[0] - 0.5 * dx[0]
    edges[-1] = x[-1] + 0.5 * dx[-1]
    return edges

P_AU_SI = sc.physical_constants["atomic unit of momentum"][0]
M_ION_KG = 4.0 * sc.atomic_mass

def p_au_to_E_eV(p_au):
    p_si = np.asarray(p_au, dtype=float) * P_AU_SI
    return p_si**2 / (2.0 * M_ION_KG) / sc.e

def single_cluster_kedi(E, Emax):
    """
    Saalmann single-cluster spectrum:
        dP/dE = (3/2Emax) sqrt(E/Emax), 0 < E < Emax
    """
    E = np.asarray(E, dtype=float)
    Emax = np.asarray(Emax, dtype=float)
    Emax = np.maximum(Emax, 1e-30)

    eps = np.clip(E[:, None] / Emax[None, :], 0.0, None)
    return (1.5 / Emax[None, :]) * np.sqrt(eps) * (eps <= 1.0)

def mixture_saalmann(E_grid, Emax_all, weight_all, chunk=2000):
    """
    Weighted sum of many Saalmann single-cluster spectra.
    """
    y = np.zeros_like(E_grid, dtype=float)
    n = len(Emax_all)

    for i0 in range(0, n, chunk):
        i1 = min(i0 + chunk, n)
        Y = single_cluster_kedi(E_grid, Emax_all[i0:i1])
        y += np.sum(Y * weight_all[i0:i1][None, :], axis=1)

    return normalize_area(y, E_grid)

def read_exp_energy_distribution(h5_path, run_no):
    """
    Read experimental ion energy distribution directly if available.
    Otherwise convert momentum histogram to dN/dE.
    """
    with h5py.File(h5_path, "r") as h5:
        hist = h5[f"runs/run_{run_no}/hist"]

        candidates = [
            ("edges_E_eV",  "centers_E_eV",  "counts_E"),
            ("edges_E",     "centers_E",     "counts_E"),
            ("edges_KE_eV", "centers_KE_eV", "counts_KE"),
            ("edges_KE",    "centers_KE",    "counts_KE"),
            ("E_edges",     "E_centers",     "counts_E"),
            ("energy_edges","energy_centers","counts_E"),
        ]

        for edge_key, center_key, count_key in candidates:
            if center_key in hist and count_key in hist:
                E_centers = hist[center_key][:].astype(float)
                counts_E = hist[count_key][:].astype(float)

                if edge_key in hist:
                    E_edges = hist[edge_key][:].astype(float)
                else:
                    E_edges = centers_to_edges(E_centers)

                dE = np.diff(E_edges)
                dE[dE <= 0] = np.nan
                dNdE = counts_E / dE
                return E_centers, dNdE, f"direct energy histogram: {center_key}, {count_key}"

        if "centers_P" not in hist or "counts_P" not in hist:
            raise KeyError("Need either energy histogram or momentum histogram centers_P/counts_P.")

        p_centers = hist["centers_P"][:].astype(float)
        counts_P = hist["counts_P"][:].astype(float)

        if "edges_P" in hist:
            p_edges = hist["edges_P"][:].astype(float)
        else:
            p_edges = centers_to_edges(p_centers)

        p_edges = np.clip(p_edges, 0.0, None)
        E_edges = p_au_to_E_eV(p_edges)
        E_centers = 0.5 * (E_edges[1:] + E_edges[:-1])

        dE = np.diff(E_edges)
        dE[dE <= 0] = np.nan
        dNdE = counts_P / dE

        return E_centers, dNdE, "converted from momentum histogram using Jacobian"

def build_fixed_size_distribution():
    """
    Fixed log-normal helium droplet size distribution.
    This is not fitted here.
    """
    lnN0 = np.log(N0_CLUSTER)
    lnN_min = lnN0 - N_SIGMA_RANGE * NU_SIZE_FIXED
    lnN_max = lnN0 + N_SIGMA_RANGE * NU_SIZE_FIXED

    lnN_grid = np.linspace(lnN_min, lnN_max, N_SIZE_GRID)
    N_grid = np.exp(lnN_grid)
    dlnN = lnN_grid[1] - lnN_grid[0]

    # log-normal in N
    w_N = np.exp(-0.5 * ((lnN_grid - lnN0) / NU_SIZE_FIXED)**2) * dlnN
    w_N /= np.sum(w_N)

    return N_grid, w_N

def saalmann_combined_spectrum(E_grid, mean_final_energy_eV, I_min_rel):
    """
    Saalmann model averaged over:
      - fixed log-normal cluster size distribution
      - Gaussian laser transverse profile
    """
    # fixed size distribution
    N_grid, w_N = build_fixed_size_distribution()
    factor_N = (N_grid / N0_CLUSTER)**(2.0 / 3.0)

    # laser profile
    rho_max_um = LASER_WAIST_UM * np.sqrt(-0.5 * np.log(I_min_rel))
    rho_edges_um = np.linspace(0.0, rho_max_um, N_LASER_GRID + 1)
    rho_mid_um = 0.5 * (rho_edges_um[1:] + rho_edges_um[:-1])
    drho_um = np.diff(rho_edges_um)

    I_rel = np.exp(-2.0 * (rho_mid_um / LASER_WAIST_UM)**2)

    # clusters uniformly sample transverse area
    w_I = rho_mid_um * drho_um
    w_I /= np.sum(w_I)

    factor_I = I_rel**GAMMA_INTENSITY

    factor_all = (factor_N[:, None] * factor_I[None, :]).ravel()
    weight_all = (w_N[:, None] * w_I[None, :]).ravel()

    mean_factor = np.sum(weight_all * factor_all)

    # <E> = (3/5) * Emax_ref * <factor>
    Emax_ref = mean_final_energy_eV / ((3.0 / 5.0) * mean_factor)
    Emax_all = Emax_ref * factor_all

    y = mixture_saalmann(E_grid, Emax_all, weight_all, chunk=2000)

    info = {
        "N_grid": N_grid,
        "w_N": w_N,
        "rho_mid_um": rho_mid_um,
        "w_I": w_I,
        "I_rel": I_rel,
        "rho_max_um": rho_max_um,
        "Emax_ref_eV": Emax_ref,
        "mean_factor": mean_factor,
        "mean_check_eV": trapz_compat(E_grid * y, E_grid),
    }
    return y, info

# ---------------------------------------------------------
# Read experiment
# ---------------------------------------------------------
E_exp, dNdE_exp, exp_source = read_exp_energy_distribution(EXP_ION_H5, RUN_EXP_ION)

m = np.isfinite(E_exp) & np.isfinite(dNdE_exp) & (E_exp >= 0) & (dNdE_exp >= 0)
E_exp = E_exp[m]
dNdE_exp = dNdE_exp[m]

order = np.argsort(E_exp)
E_exp = E_exp[order]
dNdE_exp = dNdE_exp[order]

# ---------------------------------------------------------
# Model energy grid
# ---------------------------------------------------------
E_edges = np.linspace(0.0, E_MAX_EV, N_E_BINS + 1)
E_mid = 0.5 * (E_edges[1:] + E_edges[:-1])

# ---------------------------------------------------------
# Build model curves for several I_min/I0
# ---------------------------------------------------------
model_results = []

for I_min_rel in I_MIN_REL_VALUES:
    y_model, info = saalmann_combined_spectrum(
        E_grid=E_mid,
        mean_final_energy_eV=MEAN_FINAL_ENERGY_EV,
        I_min_rel=I_min_rel,
    )

    model_results.append({
        "I_min_rel": I_min_rel,
        "y_model": y_model,
        "info": info,
    })

# ---------------------------------------------------------
# Print summary
# ---------------------------------------------------------
print("Saalmann model with fixed helium droplet size distribution")
print("----------------------------------------------------------")
print(f"Experimental source: {exp_source}")
print()
print("Fixed source parameters:")
print(f"  nozzle diameter = {NOZZLE_DIAMETER_UM:.1f} um")
print(f"  T0 = {NOZZLE_T_K:.1f} K")
print(f"  P0 = {STAGNATION_PRESSURE_BAR:.1f} bar")
print(f"  N0 fixed = {N0_CLUSTER:.3e} atoms")
print(f"  log-normal width nu fixed = {NU_SIZE_FIXED:.3f}")
print()
print("Manual global parameters:")
print(f"  mean final energy = {MEAN_FINAL_ENERGY_EV:.3f} eV")
print(f"  gamma = {GAMMA_INTENSITY:.3f}")
print()

for res in model_results:
    Imin = res["I_min_rel"]
    info = res["info"]
    print(f"I_min/I0 = {Imin:.4g}")
    print(f"  I_min = {Imin * I0_WCM2:.3e} W/cm^2")
    print(f"  rho_max = {info['rho_max_um']:.3f} um")
    print(f"  Emax_ref = {info['Emax_ref_eV']:.4f} eV")
    print(f"  mean check = {info['mean_check_eV']:.4f} eV")
    print()

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))

# Linear plot
ax = axes[0]
ax.plot(
    E_exp,
    normalize_max(dNdE_exp),
    "o",
    ms=4,
    alpha=0.70,
    label=f"Experiment run {RUN_EXP_ION}",
)

for res in model_results:
    Imin = res["I_min_rel"]
    y_model = res["y_model"]

    lw = 3.0 if np.isclose(Imin, BEST_I_MIN_REL) else 2.0
    alpha = 1.0 if np.isclose(Imin, BEST_I_MIN_REL) else 0.80

    ax.plot(
        E_mid,
        normalize_max(y_model),
        lw=lw,
        alpha=alpha,
        label=rf"Saalmann, $I_{{min}}/I_0={Imin:.3g}$",
    )

ax.set_xlabel("Ion kinetic energy (eV)")
ax.set_ylabel("Normalized signal")
ax.set_xlim(0, E_MAX_EV)
ax.set_ylim(0, 1.08)
ax.set_title("Saalmann model + fixed He droplet size distribution")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

# Log plot
ax = axes[1]
ax.plot(
    E_exp,
    np.maximum(normalize_max(dNdE_exp), 1e-6),
    "o",
    ms=4,
    alpha=0.70,
    label="Experiment",
)

for res in model_results:
    Imin = res["I_min_rel"]
    y_model = res["y_model"]

    lw = 3.0 if np.isclose(Imin, BEST_I_MIN_REL) else 2.0
    alpha = 1.0 if np.isclose(Imin, BEST_I_MIN_REL) else 0.80

    ax.plot(
        E_mid,
        np.maximum(normalize_max(y_model), 1e-6),
        lw=lw,
        alpha=alpha,
        label=rf"$I_{{min}}/I_0={Imin:.3g}$",
    )

ax.set_xlabel("Ion kinetic energy (eV)")
ax.set_ylabel("Normalized signal")
ax.set_xlim(0, E_MAX_EV)
ax.set_yscale("log")
ax.set_ylim(1e-4, 2.0)
ax.set_title("Same comparison, log scale")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# Save result
# ---------------------------------------------------------
saalmann_fixed_source_result = {
    "E_exp": E_exp,
    "dNdE_exp": dNdE_exp,
    "E_mid": E_mid,
    "model_results": model_results,
    "MEAN_FINAL_ENERGY_EV": MEAN_FINAL_ENERGY_EV,
    "I_MIN_REL_VALUES": I_MIN_REL_VALUES,
    "BEST_I_MIN_REL": BEST_I_MIN_REL,
    "GAMMA_INTENSITY": GAMMA_INTENSITY,
    "N0_CLUSTER": N0_CLUSTER,
    "NU_SIZE_FIXED": NU_SIZE_FIXED,
    "NOZZLE_DIAMETER_UM": NOZZLE_DIAMETER_UM,
    "NOZZLE_T_K": NOZZLE_T_K,
    "STAGNATION_PRESSURE_BAR": STAGNATION_PRESSURE_BAR,
    "LASER_WAIST_UM": LASER_WAIST_UM,
    "I0_WCM2": I0_WCM2,
    "exp_source": exp_source,
}

print("Saved dictionary: saalmann_fixed_source_result")
